# Notebook B: Supply Chain Construction

Builds the full supply chain edge table: processing stage classification,
copper edges (mine to smelter to port), export destination edges (port to country),
and non-copper mineral edges (Fe, Li, Mo, Au, Ag, I, etc.).

**Reads:** `_pipeline_state_2.pkl` (from Notebook A)  
**Writes:** `_pipeline_state_5.pkl`  
**CSVs output:** `Chile_Ports.csv`, `Chile_Supply_Chain_Edges.csv`, `Chile_Export_Destinations.csv`,
`Port_Product_Shares.csv` (Aduanas-derived, if Salidas file is available)

**Combines:** original Parts 2, 3, 4  
**Depends on:** Notebook A  


In [1]:
import sys
import os

# Find the Chile project root (contains scripts/ and inputs/) by walking up from cwd.
# Works whether you start Jupyter from Chile/ or from pipeline/.
def _find_chile_root():
    d = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(d, "scripts")) and os.path.isdir(os.path.join(d, "inputs")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError(
        "Could not find Chile project root (expected to contain scripts/ and inputs/)."

    )

_utils_dir = os.path.join(_find_chile_root(), "scripts")
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

from pipeline_utils import *
from pipeline_constants import *  # MO_SPLIT, FALLBACK_SHARES, ADUANAS_PORT_ALIAS, FE_PORTS_EXTRA, …

state = load_state(2)
inv, links, comm_col, idle_mines, edges, ports_df = unpack_state(state)
MATCH_DISAMBIGUATION = state['MATCH_DISAMBIGUATION']
IRON_MINE_NAMES = state['IRON_MINE_NAMES']
ZINC_MINE_NAMES = state['ZINC_MINE_NAMES']
commodity_prices = state['commodity_prices']

print(f"Loaded state from Notebook A: {len(inv)} inv rows, {len(links)} link rows")


Loaded state from Notebook A: 461 inv rows, 1109 link rows


## 3: Copper Supply Chain Edges

Processing stage classification, smelter/port setup, product form assignment, and unified edge table.

In [2]:

# 3A. Processing stage classification
# USGS/SNL FACILITY_TYPE is coarse: "Processing Plant" covers concentrators,
# refineries, SX-EW plants, and smelters alike. A second keyword pass on the
# facility name is needed to split these into the distinct chain stages.

section_header("3A. PROCESSING STAGE CLASSIFICATION")

STAGE_MAP = {
    "Mine (active)": "extraction", "Mine (idle)": "extraction_idle",
    "Mine (USGS)": "extraction", "Prospect/Project": "extraction",
    "Concentrator": "concentration", "SX-EW Plant": "sx_ew",
    "Smelter": "smelting", "Refinery": "refining",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}
inv["CHAIN_STAGE"] = inv["FACILITY_TYPE"].map(STAGE_MAP).fillna("other")

# Refine ambiguous "processing" assignments using name keywords.
# Order matters: smelting before refining before sx_ew to avoid mis-assigning
# combined facilities (e.g. "Las Ventanas refinery and smelter").
proc_mask = inv["CHAIN_STAGE"] == "processing"
name_lower = inv["FACILITY_NAME"].str.lower()
inv.loc[proc_mask & name_lower.str.contains("smelter|fundici|smelting", na=False), "CHAIN_STAGE"] = "smelting"
inv.loc[proc_mask & name_lower.str.contains("refin|electro", na=False) & (inv["CHAIN_STAGE"] == "processing"), "CHAIN_STAGE"] = "refining"
inv.loc[proc_mask & name_lower.str.contains("sx-ew|sx/ew|leach|lixiv|cathode|catod", na=False) & (inv["CHAIN_STAGE"] == "processing"), "CHAIN_STAGE"] = "sx_ew"
inv.loc[proc_mask & name_lower.str.contains("concentrat|flotation|mill", na=False) & (inv["CHAIN_STAGE"] == "processing"), "CHAIN_STAGE"] = "concentration"

for stage, count in inv["CHAIN_STAGE"].value_counts().items():
    print(f"  {stage:<18} {count:>4}")

# 3B. Smelter matching to inventory

section_header("3B. SMELTER AND PORT SETUP")

smelter_inv_map = {}
for sm in SMELTERS:
    for term in sm["search"]:
        hits = inv[inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)]
        if len(hits) > 0:
            idx = hits.index[0]
            smelter_inv_map[sm["name"]] = inv.at[idx, "FACILITY_NAME"]
            inv.at[idx, "CHAIN_STAGE"] = "smelting"
            print(f"  {sm['name']:<30} -> {inv.at[idx, 'FACILITY_NAME']}")
            break
    else:
        print(f"  {sm['name']:<30} -> NOT IN INVENTORY")

ports_df = pd.DataFrame(PORTS)
ports_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Ports.csv"), index=False)
print(f"\n  Ports saved: {len(ports_df)}")

# 3C. Product form assignment (vectorized)

section_header("3C. PRODUCT FORM + DOWNSTREAM EDGES")

pt = links.get("PLANT_TYPE", pd.Series("", index=links.index)).str.lower()
pn = links.get("PLANT_NAME", pd.Series("", index=links.index)).str.lower()

conditions = [
    pt.str.contains("sx-ew|sx/ew", na=False) | pn.str.contains("sx-ew|sx/ew|leach|cathode", na=False),
    pt.str.contains("smelter", na=False) | pn.str.contains("smelter|fundici", na=False),
    pt.str.contains("refin", na=False) | pn.str.contains("refin|electro", na=False),
    pt.str.contains("concentrat", na=False) | pn.str.contains("concentrat|flotation|mill", na=False),
]
choices = ["cathode_sxew", "blister", "cathode_er", "concentrate"]
links["PRODUCT_FORM"] = np.select(conditions, choices, default="concentrate")

# Concentrate is the default because it is the most common Cu output form;
# unmatched links are most likely copper concentrate producers.
n_defaulted = (~np.any(conditions, axis=0)).sum()
print("Product forms in mine-plant links:")
print(links["PRODUCT_FORM"].value_counts().to_string())
if n_defaulted > 0:
    print(f"\n  Note: {n_defaulted} links defaulted to 'concentrate' (no keyword match)")

# 3D. Build downstream edges

downstream_edges = []
concentrators = inv[inv["CHAIN_STAGE"] == "concentration"].copy()
smelter_fed = set()

# A. Concentrator -> Smelter (named feed mines + regional feed)
for sm in SMELTERS:
    sm_inv_name = smelter_inv_map.get(sm["name"], sm["name"])

    # Named feeds
    for mine_term in sm.get("feeds_from_mines", []):
        concs = concentrators[concentrators["FACILITY_NAME"].str.contains(
            mine_term, case=False, na=False, regex=False)]
        for cidx, crow in concs.iterrows():
            downstream_edges.append({
                "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
                "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
                "TO_NAME": sm_inv_name, "TO_TYPE": "smelter",
                "TO_LAT": sm["lat"], "TO_LON": sm["lon"],
                "EDGE_TYPE": "concentrate_to_smelter", "PRODUCT_FORM": "concentrate",
                "OPERATOR": sm["operator"],
                "DISTANCE_KM": haversine_km(crow["LATITUD"], crow["LONGITUD"], sm["lat"], sm["lon"])
                if pd.notna(crow["LATITUD"]) else None,
            })
            smelter_fed.add(cidx)

    # Regional feed for custom smelters (Altonorte, Paipote)
    if sm.get("feeds_from_region") and not sm.get("feeds_from_mines"):
        region = sm["feeds_from_region"]
        regional_concs = concentrators[
            concentrators.get("REGION", pd.Series("", index=concentrators.index)).str.contains(region, case=False, na=False) &
            ~concentrators.index.isin(smelter_fed)
        ]
        for cidx, crow in regional_concs.iterrows():
            if pd.isna(crow["LATITUD"]): continue
            dist = haversine_km(crow["LATITUD"], crow["LONGITUD"], sm["lat"], sm["lon"])
            if dist <= 300:
                downstream_edges.append({
                    "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
                    "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
                    "TO_NAME": sm_inv_name, "TO_TYPE": "smelter",
                    "TO_LAT": sm["lat"], "TO_LON": sm["lon"],
                    "EDGE_TYPE": "concentrate_to_smelter", "PRODUCT_FORM": "concentrate",
                    "OPERATOR": sm["operator"], "DISTANCE_KM": round(dist, 1),
                })
                smelter_fed.add(cidx)

# B. Smelter -> Port
for sm in SMELTERS:
    sm_inv_name = smelter_inv_map.get(sm["name"], sm["name"])
    for port_name in sm["export_ports"]:
        port = next((p for p in PORTS if p["name"] == port_name), None)
        if port:
            downstream_edges.append({
                "FROM_NAME": sm_inv_name, "FROM_TYPE": "smelter",
                "FROM_LAT": sm["lat"], "FROM_LON": sm["lon"],
                "TO_NAME": port["name"], "TO_TYPE": "port",
                "TO_LAT": port["lat"], "TO_LON": port["lon"],
                "EDGE_TYPE": "smelter_to_port", "PRODUCT_FORM": sm["output_product"],
                "OPERATOR": sm["operator"],
                "DISTANCE_KM": haversine_km(sm["lat"], sm["lon"], port["lat"], port["lon"]),
            })

# C. Concentrator -> Port (dedicated overrides first, then nearest)
for cidx, crow in concentrators.iterrows():
    if pd.isna(crow["LATITUD"]) or cidx in smelter_fed:
        continue
    # Check for dedicated port override
    facility_name = str(crow["FACILITY_NAME"]).lower()
    assigned_port = None
    for key, port_name in DEDICATED_PORT.items():
        if key.lower() in facility_name:
            port = next((p for p in PORTS if p["name"] == port_name), None)
            if port:
                assigned_port = port
                dist = haversine_km(crow["LATITUD"], crow["LONGITUD"], port["lat"], port["lon"])
                break
    if not assigned_port:
        assigned_port, dist = nearest_port(crow["LATITUD"], crow["LONGITUD"], "concentrate")
    if assigned_port:
        downstream_edges.append({
            "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
            "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
            "TO_NAME": assigned_port["name"], "TO_TYPE": "port",
            "TO_LAT": assigned_port["lat"], "TO_LON": assigned_port["lon"],
            "EDGE_TYPE": "concentrate_to_port", "PRODUCT_FORM": "concentrate",
            "OPERATOR": crow.get("OPERATOR_NAME", ""),
            "DISTANCE_KM": round(dist, 1) if isinstance(dist, (int, float)) else None,
        })

# D. SX-EW -> Port (Codelco consolidation first, then nearest cathode port)
for pidx, prow in inv[inv["CHAIN_STAGE"] == "sx_ew"].iterrows():
    if pd.isna(prow["LATITUD"]): continue
    facility_name = str(prow["FACILITY_NAME"]).lower()
    # Check Codelco cathode routing override
    assigned_port = None
    for key, port_name in CODELCO_CATHODE_ROUTING.items():
        if key.lower() in facility_name:
            port = next((p for p in PORTS if p["name"] == port_name), None)
            if port:
                assigned_port = port
                dist = haversine_km(prow["LATITUD"], prow["LONGITUD"], port["lat"], port["lon"])
                break
    # Also check dedicated port overrides
    if not assigned_port:
        for key, port_name in DEDICATED_PORT.items():
            if key.lower() in facility_name:
                port = next((p for p in PORTS if p["name"] == port_name), None)
                if port:
                    assigned_port = port
                    dist = haversine_km(prow["LATITUD"], prow["LONGITUD"], port["lat"], port["lon"])
                    break
    if not assigned_port:
        assigned_port, dist = nearest_port(prow["LATITUD"], prow["LONGITUD"], "cathode")
    if assigned_port:
        downstream_edges.append({
            "FROM_NAME": prow["FACILITY_NAME"], "FROM_TYPE": "sx_ew",
            "FROM_LAT": prow["LATITUD"], "FROM_LON": prow["LONGITUD"],
            "TO_NAME": assigned_port["name"], "TO_TYPE": "port",
            "TO_LAT": assigned_port["lat"], "TO_LON": assigned_port["lon"],
            "EDGE_TYPE": "sxew_to_port", "PRODUCT_FORM": "cathode_sxew",
            "OPERATOR": prow.get("OPERATOR_NAME", ""),
            "DISTANCE_KM": round(dist, 1),
        })

for etype in ["concentrate_to_smelter", "smelter_to_port", "concentrate_to_port", "sxew_to_port"]:
    n = sum(1 for e in downstream_edges if e["EDGE_TYPE"] == etype)
    print(f"  {etype:<25} {n:>4}")

# 3E. Unified edge table
# Anti-cycle guard: upstream edges are strictly mine->plant (FROM_TYPE=mine,
# TO_TYPE=plant). Downstream edges are concentrator/sx_ew->port or
# concentrator->smelter->port. No plant->mine reverse edges are ever created,
# so cycles cannot form by construction. The FROM_TYPE/TO_TYPE columns enforce
# this: "mine" nodes only appear as FROM, "port" nodes only as TO.

section_header("3E. UNIFIED SUPPLY CHAIN EDGES")

upstream = links[["MINE_NAME", "PLANT_NAME", "MINE_LAT", "MINE_LON",
                  "PLANT_LAT", "PLANT_LON", "SHARED_COMMODITIES",
                  "DISTANCE_KM", "PRODUCT_FORM"]].copy()
upstream.rename(columns={
    "MINE_NAME": "FROM_NAME", "PLANT_NAME": "TO_NAME",
    "MINE_LAT": "FROM_LAT", "MINE_LON": "FROM_LON",
    "PLANT_LAT": "TO_LAT", "PLANT_LON": "TO_LON",
    "SHARED_COMMODITIES": "COMMODITIES",
}, inplace=True)
upstream["FROM_TYPE"] = "mine"
upstream["TO_TYPE"] = "plant"
upstream["EDGE_TYPE"] = "mine_to_plant"

downstream_df = pd.DataFrame(downstream_edges)

# Enrich downstream commodities from inventory (instead of blanket "Copper")
def get_facility_commodities(facility_name):
    matches = inv[inv["FACILITY_NAME"] == facility_name]
    if len(matches) == 0:
        # Use longer prefix (min 16 chars) and verify single match to avoid
        # cross-contamination (e.g. "Cerro Negro" matching "Cerro Negro Norte")
        # Truncate to 16 chars so that e.g. "Escondida Mine" can match
        # "Escondida concentrator".  Using max(16, len) was a bug: when the
        # name is >= 16 chars max() returns the full length, making the
        # substring search identical to the exact search that already failed.
        prefix = facility_name[:16]
        matches = inv[inv["FACILITY_NAME"].str.contains(prefix, case=False, na=False, regex=False)]
        if len(matches) > 1:
            # Ambiguous: pick exact facility type match if possible
            type_match = matches[matches["CHAIN_STAGE"].isin(["concentration", "smelting", "sx_ew"])]
            if len(type_match) > 0:
                matches = type_match
    if len(matches) == 0:
        return "Copper"
    comms = parse_comm_list(matches.iloc[0].get(comm_col, ""))
    if not comms: return "Copper"
    cu_prod = matches.iloc[0].get("COCHILCO_CU_2024_KMT", None)
    if (cu_prod is not None and pd.notna(cu_prod) and cu_prod > 0) or "Copper" in comms:
        return "Copper"
    return comms[0]

downstream_df["COMMODITIES"] = downstream_df["FROM_NAME"].apply(get_facility_commodities)
downstream_df["DISTANCE_KM"] = downstream_df["DISTANCE_KM"].round(1)

common_cols = ["FROM_NAME", "FROM_TYPE", "FROM_LAT", "FROM_LON",
               "TO_NAME", "TO_TYPE", "TO_LAT", "TO_LON",
               "EDGE_TYPE", "PRODUCT_FORM", "COMMODITIES", "DISTANCE_KM"]
for col in common_cols:
    for df in [upstream, downstream_df]:
        if col not in df.columns:
            df[col] = ""

edges = pd.concat([upstream[common_cols], downstream_df[common_cols]], ignore_index=True)

print(f"Unified edges: {len(edges)}")
for et, count in edges["EDGE_TYPE"].value_counts().items():
    print(f"  {et:<25} {count:>5}")


3A. PROCESSING STAGE CLASSIFICATION
  extraction          187
  processing          129
  extraction_idle      80
  sx_ew                25
  concentration        22
  smelting             14
  refining              4

3B. SMELTER AND PORT SETUP
  Chuquicamata smelter           -> Chuquicamata SX-EW plant (oxide) and smelter
  Potrerillos smelter            -> Potrerillos SX-EW refinery and smelter
  Caletones smelter              -> El Teniente plant
  Altonorte smelter              -> Altonorte smelter
  Paipote smelter (H.V. Lira)    -> Hernán Videla Lira smelter
  Chagres smelter                -> Chagres smelter

  Ports saved: 13

3C. PRODUCT FORM + DOWNSTREAM EDGES
Product forms in mine-plant links:
PRODUCT_FORM
cathode_sxew    571
concentrate     440
blister          82
cathode_er       16

  Note: 342 links defaulted to 'concentrate' (no keyword match)
  concentrate_to_smelter      10
  smelter_to_port              9
  concentrate_to_port         12
  sxew_to_port            

## 4: Export Destinations

Three-tier export edge construction:
- **4A.** Aduanas Salidas data for port-level shares and port-country flows
- **4B.** Comtrade cross-validation against Aduanas totals
- **4C.** Comtrade HS6 exports for country-level destinations across all minerals

Priority for edge building: Aduanas port-country (most granular) > Comtrade + port shares > COCHILCO + port shares.
Copper keeps COCHILCO product-form splits (concentrate/cathode/blister). All other minerals use Comtrade as primary source.


In [3]:

section_header("4. EXPORT DESTINATIONS")

wb = openpyxl.load_workbook(COCHILCO_ORIG, read_only=True, data_only=True)

def parse_destination_table(sheet_name, target_year=2024):
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    yr_row, yc = None, None
    for i, row in enumerate(rows):
        if sum(1 for v in row if isinstance(v, (int, float)) and 2004 < v < 2035) >= 5:
            yr_row = i
            for j, v in enumerate(row):
                if isinstance(v, (int, float)) and int(v) == target_year:
                    yc = j
            break
    if yr_row is None or yc is None:
        print(f"  WARNING: Could not find {target_year} in {sheet_name}")
        return {}
    result = {}
    current_region = None
    for i in range(yr_row + 1, len(rows)):
        row = rows[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        val = row[yc] if yc < len(row) else None
        if not label or label.startswith("(") or label.startswith("Fuente") or label.startswith("Source"):
            continue
        if any(r in label for r in ["EUROPA", "AMÉRICA", "ASIA", "OCEANÍA", "AFRICA"]):
            current_region = label.split("/")[0].strip()
            continue
        if label in ("TOTAL", "OTROS / Other"):
            continue
        if isinstance(val, (int, float)) and val > 0:
            parts = label.split("/")
            country = parts[-1].strip() if len(parts) > 1 else parts[0].strip()
            result[country] = {"value": val, "region": current_region or ""}
    return result

def parse_nonmetallic_column(sheet_name, col_idx, target_rows=(10, 65), expected_header=None):
    """Parse a column from a non-metallic export table.
    If expected_header is provided, verify the column header matches before reading."""
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    # Optional header validation: check that the column contains the expected label
    if expected_header is not None:
        found = False
        for ri in range(min(10, len(rows))):
            cell_val = rows[ri][col_idx] if col_idx < len(rows[ri]) else None
            if cell_val is not None and expected_header.upper() in str(cell_val).upper():
                found = True
                break
        if not found:
            print(f"  WARNING: expected header '{expected_header}' not found in "
                  f"{sheet_name} col {col_idx}. Data may be misaligned.")
    result, current_region = {}, None
    for i in range(target_rows[0], min(target_rows[1], len(rows))):
        row = rows[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        val = row[col_idx] if col_idx < len(row) else None
        if not label: continue
        if any(r in label for r in ["EUROPA", "AMÉRICA", "ASIA"]):
            current_region = label.split("/")[0].strip()
            continue
        if label in ("TOTAL", "TOTAL MINERÍA"): continue
        if isinstance(val, (int, float)) and val > 0:
            parts = label.split("/")
            country = parts[-1].strip() if len(parts) > 1 else parts[0].strip()
            result[country] = {"value": val, "region": current_region or ""}
    return result

print("Parsing export destination tables...\n")
cu_refined = parse_destination_table("Tabla 18.2")
cu_blister = parse_destination_table("Tabla 19.2")
cu_concentrate = parse_destination_table("Tabla 20.2")
mo_concentrate = parse_destination_table("Tabla 23.2")
li_exports = parse_nonmetallic_column("Tabla 11", col_idx=2, expected_header="LITIO")
io_exports = parse_nonmetallic_column("Tabla 11", col_idx=7, expected_header="YODO")

for name, data, unit in [
    ("Cu refined", cu_refined, "kMT"), ("Cu blister", cu_blister, "kMT"),
    ("Cu concentrate", cu_concentrate, "kMT"), ("Mo concentrate", mo_concentrate, "MT"),
    ("Lithium", li_exports, "$M FOB"), ("Iodine", io_exports, "$M FOB"),
]:
    total = sum(d["value"] for d in data.values())
    print(f"  {name:<20} {len(data):>3} destinations, total: {total:>10,.1f} {unit}")

wb.close()

# Country coordinates are used only for visualisation (edge arcs on the map);
# they do not affect routing or production allocation.
COUNTRY_COORDS = {
    # Major trading partners (original)
    "China": {"lat": 31.23, "lon": 121.47}, "Japan": {"lat": 35.68, "lon": 139.69},
    "South Korea": {"lat": 37.57, "lon": 126.98}, "USA": {"lat": 29.76, "lon": -95.37},
    "Brazil": {"lat": -23.55, "lon": -46.63}, "India": {"lat": 19.08, "lon": 72.88},
    "Germany": {"lat": 53.55, "lon": 9.99}, "Spain": {"lat": 36.72, "lon": -4.42},
    "France": {"lat": 48.86, "lon": 2.35}, "Italy": {"lat": 45.46, "lon": 9.19},
    "Netherlands": {"lat": 51.92, "lon": 4.48}, "Belgium": {"lat": 51.26, "lon": 4.35},
    "Sweden": {"lat": 57.71, "lon": 11.97}, "Bulgaria": {"lat": 42.70, "lon": 23.32},
    "Finland": {"lat": 60.17, "lon": 24.94}, "Canada": {"lat": 49.28, "lon": -123.12},
    "Mexico": {"lat": 19.43, "lon": -99.13}, "Taiwan": {"lat": 25.03, "lon": 121.57},
    "Thailand": {"lat": 13.76, "lon": 100.50}, "Philippines": {"lat": 14.60, "lon": 120.98},
    "Malaysia": {"lat": 3.14, "lon": 101.69}, "Indonesia": {"lat": -6.21, "lon": 106.85},
    "Vietnam": {"lat": 10.82, "lon": 106.63}, "Peru": {"lat": -12.05, "lon": -77.04},
    "Colombia": {"lat": 4.71, "lon": -74.07}, "Argentina": {"lat": -34.60, "lon": -58.38},
    "Turkey": {"lat": 41.01, "lon": 28.98}, "United Kingdom": {"lat": 51.51, "lon": -0.13},
    "Switzerland": {"lat": 47.38, "lon": 8.54}, "Singapore": {"lat": 1.35, "lon": 103.82},
    "Greece": {"lat": 37.98, "lon": 23.73}, "Portugal": {"lat": 38.72, "lon": -9.14},
    "Panama": {"lat": 8.98, "lon": -79.52}, "Bahrain": {"lat": 26.07, "lon": 50.56},
    "UAE": {"lat": 25.20, "lon": 55.27}, "Hong Kong": {"lat": 22.32, "lon": 114.17},
    "Poland": {"lat": 52.23, "lon": 21.01}, "Norway": {"lat": 59.91, "lon": 10.75},
    # Extended from Comtrade export partners
    "Costa Rica": {"lat": 9.93, "lon": -84.09}, "Cambodia": {"lat": 11.56, "lon": 104.92},
    "South Africa": {"lat": -33.92, "lon": 18.42}, "Namibia": {"lat": -22.56, "lon": 17.08},
    "Bangladesh": {"lat": 23.81, "lon": 90.41}, "Bolivia": {"lat": -16.50, "lon": -68.15},
    "Ecuador": {"lat": -0.18, "lon": -78.47}, "Dominican Rep.": {"lat": 18.49, "lon": -69.88},
    "Paraguay": {"lat": -25.26, "lon": -57.58}, "Australia": {"lat": -33.87, "lon": 151.21},
    "Congo": {"lat": -4.27, "lon": 15.27}, "Guatemala": {"lat": 14.63, "lon": -90.51},
    "Denmark": {"lat": 55.68, "lon": 12.57}, "Uruguay": {"lat": -34.88, "lon": -56.16},
    "Honduras": {"lat": 14.07, "lon": -87.19}, "Cyprus": {"lat": 35.17, "lon": 33.36},
    "Saudi Arabia": {"lat": 24.77, "lon": 46.74}, "Austria": {"lat": 48.21, "lon": 16.37},
    "New Zealand": {"lat": -36.85, "lon": 174.76}, "Ireland": {"lat": 53.35, "lon": -6.26},
    "Nigeria": {"lat": 6.52, "lon": 3.38}, "Ghana": {"lat": 5.60, "lon": -0.19},
    "Pakistan": {"lat": 24.86, "lon": 67.01}, "Morocco": {"lat": 33.97, "lon": -6.85},
    "Jamaica": {"lat": 18.00, "lon": -76.79}, "Algeria": {"lat": 36.75, "lon": 3.04},
    "Mozambique": {"lat": -25.97, "lon": 32.57}, "Hungary": {"lat": 47.50, "lon": 19.04},
    "El Salvador": {"lat": 13.69, "lon": -89.19}, "Nicaragua": {"lat": 12.11, "lon": -86.27},
    "Venezuela": {"lat": 10.48, "lon": -66.90}, "Lebanon": {"lat": 33.89, "lon": 35.50},
    "Kuwait": {"lat": 29.38, "lon": 47.99}, "Sri Lanka": {"lat": 6.93, "lon": 79.85},
    "Israel": {"lat": 32.09, "lon": 34.77}, "Lithuania": {"lat": 54.69, "lon": 25.28},
}

COUNTRY_ALIAS = {
    "Corea del Sur": "South Korea", "Estados Unidos": "USA",
    "Emiratos Árabes Unidos": "UAE", "United Arab Emirates": "UAE",
    "Baréin": "Bahrain", "Turquía": "Turkey", "Taiwán": "Taiwan",
    "Tailandia": "Thailand", "Filipinas": "Philippines",
    "Malasia": "Malaysia", "Singapur": "Singapore",
    "Panamá": "Panama", "Noruega": "Norway",
}

def normalize_country(name):
    return COUNTRY_ALIAS.get(name, name)

# 4A. ADUANAS-DERIVED PORT SHARES

_salidas_candidates = [
    os.path.join(DIR_DATA, "Salidas2024.csv"),
    os.path.join(DIR_DATA, "salidas_2024.csv"),
    os.path.join(DIR_DATA, "Salidas2025.csv"),
    os.path.join(DIR_DATA, "Salidas_minerals_only.csv"),
]
SALIDAS_PATH = next((p for p in _salidas_candidates if os.path.exists(p)), _salidas_candidates[0])
CODIGOS_PATH = os.path.join(DIR_DATA, "tablas_de_codigos.xlsx")


HS_MINERAL_MAP = {
    "2603": ("Copper", "concentrate"), "7402": ("Copper", "blister"),
    "7403": ("Copper", "cathode"), "2613": ("Molybdenum", "mo_concentrate"),
    "2601": ("Iron", "iron_ore"), "7108": ("Gold", "gold_refined"),
    "7106": ("Silver", "silver_refined"),
    "283620": ("Lithium", "lithium_compounds"), "28362010": ("Lithium", "lithium_compounds"),
    "28362020": ("Lithium", "lithium_compounds"), "28362030": ("Lithium", "lithium_compounds"),
    "283691": ("Lithium", "lithium_compounds"), "28369100": ("Lithium", "lithium_compounds"),
    "282520": ("Lithium", "lithium_compounds"), "28252000": ("Lithium", "lithium_compounds"),
    "284290": ("Lithium", "lithium_compounds"),
    "280120": ("Iodine", "iodine"), "2528": ("Boron", "borate"),
    "2602": ("Manganese", "mn_ore"), "2834": ("Nitrate", "nitrate"),
    "284170": ("Rhenium", "perrhenate"),
}

# Port codes 827 (ZOFRI/Iquique) and 821 (Valparaíso) are absent from the standard
# Tablas de Códigos lookup; they are resolved via ADUANAS_MANUAL_PORTS.


def classify_hs(code_str):
    code = str(code_str).replace(".", "").replace(" ", "").strip()
    code_trimmed = code.rstrip("0") or code
    for prefix_len in [8, 6, 4]:
        for c in [code[:prefix_len], code_trimmed[:prefix_len]]:
            if c in HS_MINERAL_MAP:
                return HS_MINERAL_MAP[c]
    return None

def parse_codigos_sheet(path, sheet_name, key_col_name, value_col_name):
    _xl = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_row, key_idx, val_idx = None, None, None
    for i in range(min(15, len(_xl))):
        for j, v in enumerate(_xl.iloc[i].values):
            if pd.notna(v) and str(v).strip().upper() == key_col_name.upper():
                key_idx, header_row = j, i
                break
        if header_row is not None: break
    if header_row is None: return {}
    for j, v in enumerate(_xl.iloc[header_row].values):
        if pd.notna(v) and value_col_name.upper() in str(v).strip().upper():
            val_idx = j; break
    if val_idx is None: return {}
    result = {}
    for ii in range(header_row + 1, len(_xl)):
        k, v = _xl.iloc[ii, key_idx], _xl.iloc[ii, val_idx]
        if pd.isna(k): continue
        try: result[int(float(k))] = str(v).strip()
        except (ValueError, TypeError): pass
    return result

aduanas_loaded = False
aduanas_port_country = None
PORT_PRODUCT_MAP = {}

# Aduanas join key documentation
# The Salidas file uses numeric codes for ports (COD_PUERTO) and countries
# (COD_PAIS) defined in a separate Tablas_de_Codigos Excel workbook.
# Join key: mineral rows are selected by HS-6 code (arancel/item_sa column).
# Port codes are mapped via ADUANAS_PORT_ALIAS + ADUANAS_MANUAL_PORTS
# (pipeline_constants.py). Known mismatch risks:
#   - Tocopilla (nitrate port) has no standard code; appears under "Other".
#   - Free-trade zone codes 827/821 are resolved via ADUANAS_MANUAL_PORTS.
#   - Aduanas port names use all-caps Spanish; aliases normalise to mixed-case.
#   - FOB values are in Chilean pesos in some years; check fob_col for CLP vs USD.

if os.path.exists(SALIDAS_PATH):
    section_header(f"4A. ADUANAS PORT SHARES ({os.path.basename(SALIDAS_PATH)})")

    with open(SALIDAS_PATH, "r", encoding="utf-8", errors="replace") as f:
        first_line = f.readline()
    _sep = ";" if first_line.count(";") > first_line.count(",") else ","
    print(f"  Detected delimiter: {'semicolon' if _sep == ';' else 'comma'}")

    sal = pd.read_csv(SALIDAS_PATH, sep=_sep, low_memory=False, dtype=str,
                       encoding="utf-8", on_bad_lines="skip")
    sal.columns = [c.strip().upper() for c in sal.columns]
    print(f"  Salidas loaded: {len(sal):,} rows x {len(sal.columns)} cols")

    # Filter to export operations
    op_col = next((c for c in sal.columns if "TIPO_OPERACION" in c.upper()), None)
    if op_col:
        before = len(sal)
        sal = sal[sal[op_col].astype(str).str.strip().isin(EXPORT_OP_CODES)]
        print(f"  Export filter: {before:,} -> {len(sal):,} rows")

    # Map key columns
    _col_map = {}
    for c in sal.columns:
        cl = c.lower()
        if "item_sa" in cl or "arancel" in cl: _col_map.setdefault("hs", c)
        if "puerto" in cl and "embarque" in cl: _col_map.setdefault("port", c)
        if "pais" in cl and ("destino" in cl or "origen" in cl): _col_map.setdefault("country", c)
        if "fob" in cl: _col_map.setdefault("fob", c)

    required = {"hs", "port", "country", "fob"}
    if required.issubset(_col_map.keys()):
        hs_col, port_col, country_col, fob_col = _col_map["hs"], _col_map["port"], _col_map["country"], _col_map["fob"]

        sal["_hs_clean"] = sal[hs_col].astype(str).str.replace(".", "", regex=False).str.strip()
        sal["_mineral"] = sal["_hs_clean"].apply(classify_hs)
        mineral = sal[sal["_mineral"].notna()].copy()
        mineral["COMMODITY"] = mineral["_mineral"].apply(lambda x: x[0])
        mineral["PRODUCT_FORM"] = mineral["_mineral"].apply(lambda x: x[1])
        mineral["FOB_USD"] = pd.to_numeric(mineral[fob_col].str.replace(",", "."), errors="coerce").fillna(0)
        mineral["PORT_CODE"] = pd.to_numeric(mineral[port_col], errors="coerce")
        mineral["COUNTRY_CODE"] = pd.to_numeric(mineral[country_col], errors="coerce")

        print(f"  Mineral rows: {len(mineral):,} / {len(sal):,} ({len(mineral)/len(sal)*100:.1f}%)")
        print(f"  Total mineral FOB: ${mineral['FOB_USD'].sum():,.0f}")
        print(f"\n  By commodity:")
        for comm, grp in mineral.groupby("COMMODITY"):
            print(f"    {comm:<15} {len(grp):>6} rows  ${grp['FOB_USD'].sum():>15,.0f}")

        port_lookup, country_lookup, transport_lookup, region_lookup = {}, {}, {}, {}
        if os.path.exists(CODIGOS_PATH):
            country_lookup = parse_codigos_sheet(CODIGOS_PATH, "Países", "COD_PAIS", "NOMBRE_PAIS")
            port_lookup_raw = parse_codigos_sheet(CODIGOS_PATH, "Puertos", "COD_PUERTO", "NOMBRE_PUERTO")
            transport_lookup = parse_codigos_sheet(CODIGOS_PATH, "Vías de Transporte", "COD_VIA_TRANSPORTE", "NOMBRE_VIA_TRANSPORTE")
            region_lookup = parse_codigos_sheet(CODIGOS_PATH, "Regiones", "COD_REGION_ORIGEN", "NOMBRE_REGION")

            for code, raw_name in port_lookup_raw.items():
                upper = raw_name.upper().strip()
                port_lookup[code] = ADUANAS_PORT_ALIAS.get(upper, raw_name.title())
            for code, name in ADUANAS_MANUAL_PORTS.items():
                port_lookup.setdefault(code, name)

            print(f"  Codigos: {len(country_lookup)} countries, {len(port_lookup)} ports")

        mineral["PORT_NAME"] = mineral["PORT_CODE"].apply(
    lambda c: port_lookup.get(int(c)) if pd.notna(c) else None)
        for pc in mineral[mineral["PORT_NAME"].isna()]["PORT_CODE"].dropna().unique():
            vol = mineral[mineral["PORT_CODE"] == pc]["FOB_USD"].sum()
            if vol > 1_000_000:
                print(f"  Warning: unmapped port code {int(pc)}, FOB ${vol:,.0f}")

        # Transport mode summary
        via_col = next((c for c in sal.columns if "VIA_TRANSPORTE" in c.upper()), None)
        if via_col and transport_lookup:
            mineral["TRANSPORT_MODE"] = pd.to_numeric(mineral[via_col], errors="coerce").map(transport_lookup)
            print(f"\n  Transport modes (mineral FOB):")
            for mode, fob in mineral.groupby("TRANSPORT_MODE")["FOB_USD"].sum().sort_values(ascending=False).items():
                print(f"    {mode:<30} ${fob:>15,.0f}  ({fob/mineral['FOB_USD'].sum()*100:.1f}%)")

        ADUANAS_COUNTRY_ALIAS = {
            **COUNTRY_ALIAS,
            "China": "China", "Japón": "Japan", "Alemania": "Germany",
            "España": "Spain", "Francia": "France", "Italia": "Italy",
            "Países Bajos": "Netherlands", "Bélgica": "Belgium",
            "Suecia": "Sweden", "Bulgaria": "Bulgaria", "Finlandia": "Finland",
            "Canadá": "Canada", "México": "Mexico", "Perú": "Peru",
            "Argentina": "Argentina", "Brasil": "Brazil",
            "Reino Unido": "United Kingdom", "Suiza": "Switzerland",
            "Grecia": "Greece", "Portugal": "Portugal",
            "India": "India", "Indonesia": "Indonesia",
            "Hong Kong": "Hong Kong", "Polonia": "Poland",
        }
        mineral["COUNTRY_NAME"] = mineral["COUNTRY_CODE"].map(
            lambda c: country_lookup.get(int(c), f"Code_{int(c)}") if pd.notna(c) else None)
        mineral["COUNTRY_EN"] = mineral["COUNTRY_NAME"].map(
            lambda x: ADUANAS_COUNTRY_ALIAS.get(x, x) if pd.notna(x) else None)

        # Compute PORT_PRODUCT_MAP from Aduanas data.
        # This gives actual FOB-weighted port shares per product form, replacing
        # the static FALLBACK_SHARES in pipeline_constants.py wherever data exists.
        cu_mineral = mineral[mineral["COMMODITY"] == "Copper"]
        for product in ["concentrate", "cathode", "blister"]:
            prod_data = cu_mineral[(cu_mineral["PRODUCT_FORM"] == product) & cu_mineral["PORT_NAME"].notna()]
            total_fob = prod_data["FOB_USD"].sum()
            if total_fob > 0:
                shares = (prod_data.groupby("PORT_NAME")["FOB_USD"].sum() / total_fob)
                PORT_PRODUCT_MAP[product] = shares[shares >= 0.005].to_dict()

        # Non-copper commodity port shares
        for comm in mineral["COMMODITY"].unique():
            if comm == "Copper": continue
            comm_data = mineral[(mineral["COMMODITY"] == comm) & mineral["PORT_NAME"].notna()]
            total_fob = comm_data["FOB_USD"].sum()
            if total_fob > 0:
                shares = comm_data.groupby("PORT_NAME")["FOB_USD"].sum() / total_fob
                product_form = comm_data["PRODUCT_FORM"].mode().iloc[0] if len(comm_data) > 0 else "unknown"
                PORT_PRODUCT_MAP[f"{comm.lower()}_{product_form}"] = shares[shares >= 0.005].to_dict()

        print(f"\n  PORT_PRODUCT_MAP derived for: {list(PORT_PRODUCT_MAP.keys())}")
        for key, shares in PORT_PRODUCT_MAP.items():
            top3 = sorted(shares.items(), key=lambda x: -x[1])[:3]
            print(f"    {key:<30} {len(shares)} ports  (top: {', '.join(f'{p} {s*100:.1f}%' for p, s in top3)})")

        # Most granular data in the pipeline: port x country x commodity x product form.
        # Used directly in export edge building when Aduanas is available.
        aduanas_port_country = mineral[
            mineral["PORT_NAME"].notna() & mineral["COUNTRY_EN"].notna()
        ].groupby(["COMMODITY", "PRODUCT_FORM", "PORT_NAME", "COUNTRY_EN"])["FOB_USD"].sum().reset_index()
        aduanas_port_country = aduanas_port_country[aduanas_port_country["FOB_USD"] > 0]
        print(f"\n  Aduanas port-country flows: {len(aduanas_port_country)} "
              f"({aduanas_port_country['COMMODITY'].nunique()} commodities, "
              f"{aduanas_port_country['PORT_NAME'].nunique()} ports, "
              f"{aduanas_port_country['COUNTRY_EN'].nunique()} countries)")

        # Save port shares CSV
        shares_rows = [{"PRODUCT": p, "PORT": port, "FOB_SHARE": share}
                       for p in ["concentrate", "cathode", "blister"]
                       if p in PORT_PRODUCT_MAP
                       for port, share in PORT_PRODUCT_MAP[p].items()]
        if shares_rows:
            pd.DataFrame(shares_rows).to_csv(os.path.join(DIR_PRELIM, "Chile_Port_Shares_Aduanas.csv"), index=False)
            print(f"  Saved: Chile_Port_Shares_Aduanas.csv ({len(shares_rows)} rows)")

        aduanas_loaded = True
    else:
        print(f"  ERROR: Could not map required columns. Found: {_col_map}")
else:
    print(f"\n  Salidas CSV not found at {SALIDAS_PATH}, using fallback PORT_PRODUCT_MAP")

# 4B. COMTRADE CROSS-VALIDATION
# Compare Aduanas Salidas FOB totals against UN Comtrade aggregate data
# to flag discrepancies in HS classification or coverage.

COMTRADE_PATH = os.path.join(DIR_DATA, "chile_mineral_trade_combined.csv")

# HS prefix -> commodity label mapping for Comtrade aggregation
COMTRADE_HS_MAP = {
    "2603": "Copper", "7402": "Copper", "7403": "Copper",
    "2613": "Molybdenum", "2601": "Iron", "7108": "Gold", "7106": "Silver",
    "2836": "Lithium", "2825": "Lithium", "2842": "Lithium",
    "2801": "Iodine", "2528": "Boron", "2602": "Manganese",
    "2834": "Nitrate", "2841": "Rhenium",
}

if os.path.exists(COMTRADE_PATH) and aduanas_loaded:
    section_header("4B. COMTRADE vs SALIDAS CROSS-VALIDATION")

    comtrade = pd.read_csv(COMTRADE_PATH)
    ct_exp = comtrade[comtrade["flowDesc"] == "Export"].copy()
    ct_exp["hs4"] = ct_exp["cmdCode"].astype(str).str[:4]
    ct_exp["commodity"] = ct_exp["hs4"].map(COMTRADE_HS_MAP)
    ct_exp = ct_exp[ct_exp["commodity"].notna()]

    # Use latest overlapping year (Salidas is typically one calendar year)
    ct_latest = ct_exp["period"].max()
    ct_yr = ct_exp[ct_exp["period"] == ct_latest]
    print(f"  Comtrade year: {ct_latest}  |  Salidas source: {os.path.basename(SALIDAS_PATH)}")
    print(f"  Comtrade mineral export rows: {len(ct_yr)}")

    ct_agg = ct_yr.groupby("commodity").agg(
        ct_fob=("primaryValue", "sum"),
        ct_wgt_kg=("netWgt", "sum"),
        ct_partners=("partnerISO", "nunique"),
    ).sort_values("ct_fob", ascending=False)

    sal_agg = mineral.groupby("COMMODITY").agg(
        sal_fob=("FOB_USD", "sum"),
    )

    comp = ct_agg.join(sal_agg, how="outer").fillna(0)
    comp["fob_ratio"] = comp.apply(
        lambda r: r["sal_fob"] / r["ct_fob"] if r["ct_fob"] > 0 else np.nan, axis=1)
    comp["fob_diff_pct"] = comp.apply(
        lambda r: (r["sal_fob"] - r["ct_fob"]) / r["ct_fob"] * 100 if r["ct_fob"] > 0 else np.nan, axis=1)

    print(f"\n  {'Commodity':<15} {'Comtrade FOB':>18} {'Salidas FOB':>18} {'Ratio':>8} {'Diff%':>8}  {'Comtrade kg':>18}")
    print("  " + "-" * 95)
    for comm, row in comp.iterrows():
        ratio_str = f"{row['fob_ratio']:.2f}" if pd.notna(row['fob_ratio']) else "N/A"
        diff_str = f"{row['fob_diff_pct']:+.1f}%" if pd.notna(row['fob_diff_pct']) else "N/A"
        flag = ""
        if pd.notna(row['fob_ratio']) and (row['fob_ratio'] < 0.80 or row['fob_ratio'] > 1.20):
            flag = " <-- MISMATCH"
        print(f"  {comm:<15} ${row['ct_fob']:>15,.0f}  ${row['sal_fob']:>15,.0f}  {ratio_str:>8}  {diff_str:>8}{flag}")

    # Top country-level comparison for copper (largest commodity)
    print(f"\n  Copper: top destination comparison (FOB)")
    ct_cu = ct_yr[ct_yr["commodity"] == "Copper"]

    # Map Comtrade ISO3 to country names used in pipeline
    ISO3_TO_NAME = {
        "CHN": "China", "JPN": "Japan", "KOR": "South Korea", "USA": "USA",
        "BRA": "Brazil", "IND": "India", "DEU": "Germany", "ESP": "Spain",
        "FRA": "France", "ITA": "Italy", "NLD": "Netherlands", "BEL": "Belgium",
        "SWE": "Sweden", "BGR": "Bulgaria", "FIN": "Finland", "CAN": "Canada",
        "MEX": "Mexico", "TWN": "Taiwan", "THA": "Thailand", "PHL": "Philippines",
        "MYS": "Malaysia", "IDN": "Indonesia", "VNM": "Vietnam", "PER": "Peru",
        "COL": "Colombia", "ARG": "Argentina", "TUR": "Turkey", "GBR": "United Kingdom",
        "CHE": "Switzerland", "SGP": "Singapore", "GRC": "Greece", "PRT": "Portugal",
        "PAN": "Panama", "BHR": "Bahrain", "ARE": "UAE", "HKG": "Hong Kong",
        "POL": "Poland", "NOR": "Norway",
    }

    ct_cu_by_country = ct_cu.groupby("partnerISO")["primaryValue"].sum().sort_values(ascending=False).head(10)
    # Compare against Aduanas port-country flows (aggregated to country level)
    if aduanas_port_country is not None:
        sal_cu = aduanas_port_country[aduanas_port_country["COMMODITY"] == "Copper"]
        sal_cu_by_country = sal_cu.groupby("COUNTRY_EN")["FOB_USD"].sum()

        print(f"  {'Country':<20} {'Comtrade FOB':>18} {'Salidas FOB':>18} {'Ratio':>8}")
        print("  " + "-" * 70)
        for iso3, ct_val in ct_cu_by_country.items():
            country_name = ISO3_TO_NAME.get(iso3, iso3)
            sal_val = sal_cu_by_country.get(country_name, 0)
            ratio = sal_val / ct_val if ct_val > 0 else np.nan
            ratio_str = f"{ratio:.2f}" if pd.notna(ratio) else "N/A"
            print(f"  {country_name:<20} ${ct_val:>15,.0f}  ${sal_val:>15,.0f}  {ratio_str:>8}")

    comp.to_csv(os.path.join(DIR_PRELIM, "Comtrade_vs_Salidas_Validation.csv"))
    print(f"\n  Saved: Comtrade_vs_Salidas_Validation.csv")

elif not os.path.exists(COMTRADE_PATH):
    print(f"\n  Comtrade file not found at {COMTRADE_PATH}, skipping cross-validation")

# 4C. COMTRADE HS6 EXPORT DESTINATIONS
# Load the Comtrade HS6 exports file to build country-level destination
# dicts for ALL minerals. This supplements COCHILCO (which only covers
# Cu, Mo, Li, I) and Aduanas (which has port-level detail but fewer
# mineral categories in its HS classification).

COMTRADE_EXPORTS_PATH = os.path.join(DIR_DATA, "chile_mineral_trade_combined.csv")

# HS6 code -> (pipeline mineral name, product form)
COMTRADE_HS6_MAP = {
    # Copper (already covered by COCHILCO, kept for completeness)
    260300: ("Copper", "concentrate"),
    740311: ("Copper", "cathode"), 740313: ("Copper", "cathode"),
    740319: ("Copper", "cathode"),
    740200: ("Copper", "blister"),
    # Molybdenum
    261310: ("Molybdenum", "mo_concentrate"), 261390: ("Molybdenum", "mo_concentrate"),
    282570: ("Molybdenum", "mo_oxide"), 720270: ("Molybdenum", "ferro_mo"),
    284170: ("Molybdenum", "molybdate"),
    # Gold & Silver
    710812: ("Gold", "gold_refined"), 261690: ("Gold", "precious_ore"),
    710691: ("Silver", "silver_refined"),
    # Iron
    260111: ("Iron", "iron_ore"), 260112: ("Iron", "iron_pellets"),
    # Zinc
    260800: ("Zinc", "zinc_concentrate"),
    # Lithium
    283691: ("Lithium", "lithium_carbonate"), 282520: ("Lithium", "lithium_hydroxide"),
    # Iodine
    280120: ("Iodine", "iodine"), 282760: ("Iodine", "iodide_compounds"),
    282990: ("Iodine", "iodate"),
    # Nitrates
    283421: ("Nitrate", "potassium_nitrate"), 310250: ("Nitrate", "sodium_nitrate"),
    310590: ("Nitrate", "fertilizer_nec"), 310230: ("Nitrate", "ammonium_nitrate"),
    # Boron
    281000: ("Boron", "boric_acid"), 252800: ("Boron", "natural_borate"),
    # Salt
    250100: ("Salt", "salt"),
    # Potash
    310420: ("Potash", "potassium_chloride"),
    # Rhenium
    811249: ("Rhenium", "rhenium_wrought"), 811241: ("Rhenium", "rhenium_unwrought"),
    # Lead
    780191: ("Lead", "lead_unwrought"),
    # Copper sulfate
    283325: ("Copper Sulfate", "copper_sulfate"),
    # Sulfuric acid
    280700: ("Sulfuric Acid", "sulfuric_acid"),
    # Selenium
    280490: ("Selenium", "selenium"),
}

# ISO3 -> English country name (used by the pipeline)
ISO3_TO_COUNTRY = {
    "CHN": "China", "JPN": "Japan", "KOR": "South Korea", "USA": "USA",
    "BRA": "Brazil", "IND": "India", "ESP": "Spain", "NLD": "Netherlands",
    "CHE": "Switzerland", "FRA": "France", "DEU": "Germany", "CAN": "Canada",
    "ITA": "Italy", "BEL": "Belgium", "THA": "Thailand", "MEX": "Mexico",
    "PER": "Peru", "BGR": "Bulgaria", "SWE": "Sweden", "BHR": "Bahrain",
    "ARG": "Argentina", "PHL": "Philippines", "FIN": "Finland",
    "CRI": "Costa Rica", "COL": "Colombia", "MYS": "Malaysia",
    "KHM": "Cambodia", "VNM": "Vietnam", "ECU": "Ecuador",
    "ZAF": "South Africa", "NOR": "Norway", "NAM": "Namibia",
    "IDN": "Indonesia", "BGD": "Bangladesh", "GBR": "United Kingdom",
    "BOL": "Bolivia", "PAN": "Panama", "DOM": "Dominican Rep.",
    "GRC": "Greece", "PRY": "Paraguay", "ARE": "UAE", "POL": "Poland",
    "AUS": "Australia", "COG": "Congo", "SGP": "Singapore",
    "GTM": "Guatemala", "DNK": "Denmark", "URY": "Uruguay",
    "HND": "Honduras", "CYP": "Cyprus", "TUR": "Turkey",
    "HKG": "Hong Kong", "SAU": "Saudi Arabia", "AUT": "Austria",
    "NZL": "New Zealand", "IRL": "Ireland", "NGA": "Nigeria",
    "GHA": "Ghana", "PAK": "Pakistan", "PRT": "Portugal",
    "MAR": "Morocco", "JAM": "Jamaica", "DZA": "Algeria",
    "MOZ": "Mozambique", "HUN": "Hungary", "SLV": "El Salvador",
    "NIC": "Nicaragua", "VEN": "Venezuela", "LBN": "Lebanon",
    "KWT": "Kuwait", "LKA": "Sri Lanka", "ISR": "Israel",
    "LTU": "Lithuania", "TWN": "Taiwan",
}

comtrade_dests = {}  # {mineral: {country_name: {"value": USD, "region": ""}}}

if os.path.exists(COMTRADE_EXPORTS_PATH):
    section_header("4C. COMTRADE HS6 EXPORT DESTINATIONS")

    ct_raw = pd.read_csv(COMTRADE_EXPORTS_PATH)

    # Filter to exports only (file may contain imports too)
    ct_exp = ct_raw[ct_raw["flowDesc"] == "Export"].copy()
    ct_latest_yr = ct_exp["period"].max()
    ct_exp = ct_exp[ct_exp["period"] == ct_latest_yr].copy()

    # Quick import check: flag minerals where Chile is a significant net importer
    ct_imp = ct_raw[ct_raw["flowDesc"] == "Import"].copy()
    if len(ct_imp) > 0:
        ct_imp_yr = ct_imp[ct_imp["period"] == ct_imp["period"].max()]
        ct_imp_yr["hs4"] = ct_imp_yr["cmdCode"].astype(str).str[:4]
        _IMP_HS_MAP = {
            "2807": "Sulfuric Acid", "2814": "Ammonia", "2815": "Caustic Soda",
            "2523": "Cement", "3102": "Fertilizer N", "2613": "Molybdenum",
        }
        ct_imp_yr["_mineral"] = ct_imp_yr["hs4"].map(_IMP_HS_MAP)
        imp_totals = ct_imp_yr[ct_imp_yr["_mineral"].notna()].groupby("_mineral")["primaryValue"].sum()
        if len(imp_totals) > 0:
            print(f"\n  Notable mineral imports ({ct_imp['period'].max()}):")
            for mineral, val in imp_totals.sort_values(ascending=False).items():
                if val > 50e6:
                    print(f"    {mineral:<20} ${val/1e6:>8.1f}M imported")
    ct_exp["_mineral"] = ct_exp["cmdCode"].map(COMTRADE_HS6_MAP)
    ct_mineral = ct_exp[ct_exp["_mineral"].notna()].copy()
    ct_mineral["COMMODITY"] = ct_mineral["_mineral"].apply(lambda x: x[0])
    ct_mineral["PRODUCT_FORM"] = ct_mineral["_mineral"].apply(lambda x: x[1])
    ct_mineral["COUNTRY_EN"] = ct_mineral["partnerISO"].map(ISO3_TO_COUNTRY)

    print(f"  Comtrade exports: {len(ct_mineral)} rows, year {ct_latest_yr}")
    print(f"  Minerals: {ct_mineral['COMMODITY'].nunique()}, Countries: {ct_mineral['COUNTRY_EN'].nunique()}")

    # Build per-mineral destination dicts (aggregated across product forms)
    for mineral, grp in ct_mineral.groupby("COMMODITY"):
        by_country = grp[grp["COUNTRY_EN"].notna()].groupby("COUNTRY_EN")["primaryValue"].sum()
        by_country = by_country[by_country > 0].sort_values(ascending=False)
        comtrade_dests[mineral] = {
            country: {"value": val, "region": ""}
            for country, val in by_country.items()
        }

    print(f"\n  Per-mineral destination coverage:")
    for mineral, dests in sorted(comtrade_dests.items(), key=lambda x: -sum(d["value"] for d in x[1].values())):
        total = sum(d["value"] for d in dests.values())
        print(f"    {mineral:<20} {len(dests):>3} countries  ${total/1e6:>10,.1f}M")

    # Also build per-product-form dicts for minerals with multiple forms
    comtrade_dests_by_form = {}
    for (mineral, pform), grp in ct_mineral.groupby(["COMMODITY", "PRODUCT_FORM"]):
        by_country = grp[grp["COUNTRY_EN"].notna()].groupby("COUNTRY_EN")["primaryValue"].sum()
        by_country = by_country[by_country > 0]
        if len(by_country) > 0:
            comtrade_dests_by_form[(mineral, pform)] = {
                country: {"value": val, "region": ""}
                for country, val in by_country.items()
            }
else:
    print(f"\n  Comtrade exports not found at {COMTRADE_EXPORTS_PATH}")
    print("  Export edges will use COCHILCO + Aduanas only.")

# Apply fallback for missing product types
for key, fallback in PORT_PRODUCT_MAP_FALLBACK.items():
    if key not in PORT_PRODUCT_MAP:
        PORT_PRODUCT_MAP[key] = fallback

# Build port-to-country edges

def build_export_edges(dest_data, commodity, product_form, unit, port_shares):
    result = []
    port_dict = {p["name"]: p for p in ports_df.to_dict("records")}
    for country_raw, info in dest_data.items():
        country = normalize_country(country_raw)
        coords = COUNTRY_COORDS.get(country)
        if not coords: continue
        for port_name, share in port_shares.items():
            port = port_dict.get(port_name)
            if not port: continue
            result.append({
                "FROM_NAME": port_name, "FROM_TYPE": "port",
                "FROM_LAT": port["lat"], "FROM_LON": port["lon"],
                "TO_NAME": country, "TO_TYPE": "country",
                "TO_LAT": coords["lat"], "TO_LON": coords["lon"],
                "EDGE_TYPE": "port_to_country", "PRODUCT_FORM": product_form,
                "COMMODITIES": commodity, "DISTANCE_KM": None,
                "EXPORT_VALUE": round(info["value"] * share, 2),
                "EXPORT_UNIT": unit, "DESTINATION_TOTAL": round(info["value"], 2),
            })
    return result

def build_aduanas_edges(apc_df, commodity, product_form):
    sub = apc_df[(apc_df["COMMODITY"] == commodity) & (apc_df["PRODUCT_FORM"] == product_form)]
    port_dict = {p["name"]: p for p in ports_df.to_dict("records")}
    result = []
    for _, row in sub.iterrows():
        coords = COUNTRY_COORDS.get(row["COUNTRY_EN"])
        port = port_dict.get(row["PORT_NAME"])
        if not coords or not port: continue
        result.append({
            "FROM_NAME": row["PORT_NAME"], "FROM_TYPE": "port",
            "FROM_LAT": port["lat"], "FROM_LON": port["lon"],
            "TO_NAME": row["COUNTRY_EN"], "TO_TYPE": "country",
            "TO_LAT": coords["lat"], "TO_LON": coords["lon"],
            "EDGE_TYPE": "port_to_country", "PRODUCT_FORM": product_form,
            "COMMODITIES": commodity, "DISTANCE_KM": None,
            "EXPORT_VALUE": round(row["FOB_USD"], 2),
            "EXPORT_UNIT": "$FOB", "DESTINATION_TOTAL": round(row["FOB_USD"], 2),
        })
    return result

# Unified export edge construction
# Priority: (1) Aduanas port-country flows, (2) Comtrade + port shares,
#           (3) COCHILCO + port shares
#
# Copper keeps its COCHILCO product-form split (concentrate/cathode/blister)
# since Comtrade aggregates differently. All other minerals use Comtrade
# as the primary country-level source.

# COCHILCO-sourced configs (Cu product forms + minerals where COCHILCO
# has volume data that Comtrade may lack)
COCHILCO_CONFIGS = [
    ("Copper", "concentrate", cu_concentrate, "kMT", "concentrate"),
    ("Copper", "cathode", cu_refined, "kMT", "cathode"),
    ("Copper", "blister", cu_blister, "kMT", "blister"),
    ("Molybdenum", "mo_concentrate", mo_concentrate, "MT", "molybdenum_mo_concentrate"),
    ("Lithium", "lithium_compounds", li_exports, "$M_FOB", "lithium_lithium_compounds"),
    ("Iodine", "iodine", io_exports, "$M_FOB", "iodine_iodine"),
]

# Comtrade-sourced configs: every mineral from the HS6 file that is NOT
# already covered by a COCHILCO config above. Uses the aggregated
# comtrade_dests dict built in section 4C.
COMTRADE_MINERALS = [
    # (pipeline commodity name, product form for edges, port_key for share lookup)
    ("Gold",            "gold_refined",       "gold_gold_refined"),
    ("Silver",          "silver_refined",     "silver_silver_refined"),
    ("Iron",            "iron_ore",           "iron_iron_ore"),
    ("Zinc",            "zinc_concentrate",   "zinc_zinc_concentrate"),
    ("Nitrate",         "nitrate",            "nitrate_nitrate"),
    ("Boron",           "borate",             "boron_borate"),
    ("Salt",            "salt",               "salt_salt"),
    ("Potash",          "potassium_chloride", "potash_potassium_chloride"),
    ("Rhenium",         "perrhenate",         "rhenium_perrhenate"),
    ("Lead",            "lead_unwrought",     "lead_lead_unwrought"),
    ("Copper Sulfate",  "copper_sulfate",     "copper sulfate_copper_sulfate"),
    ("Sulfuric Acid",   "sulfuric_acid",      "sulfuric acid_sulfuric_acid"),
    ("Selenium",        "selenium",           "selenium_selenium"),
]

#   (Coquimbo/Los Vilos) — the correct Aduanas-2024-calibrated values
#   (Caldera/Huasco/Guayacán) are now in pipeline_constants.

export_edges = []

# Step 1: COCHILCO-based minerals (Cu forms, Mo, Li, I)
# Priority: Aduanas port-country > COCHILCO proportional
for commodity, pform, dest_data, unit, port_key in COCHILCO_CONFIGS:
    if aduanas_loaded and aduanas_port_country is not None:
        aduanas_edges = build_aduanas_edges(aduanas_port_country, commodity, pform)
        if aduanas_edges:
            export_edges.extend(aduanas_edges)
            continue
    # Fallback to COCHILCO destination data distributed by port shares
    shares = PORT_PRODUCT_MAP.get(port_key, FALLBACK_SHARES.get(port_key, {}))
    export_edges.extend(build_export_edges(dest_data, commodity, pform, unit, shares))

# Step 2: Comtrade-based minerals
# Priority: Aduanas port-country > Comtrade proportional
_comtrade_covered = set()
for commodity, pform, port_key in COMTRADE_MINERALS:
    # Try Aduanas first (has port-level granularity)
    if aduanas_loaded and aduanas_port_country is not None:
        aduanas_edges = build_aduanas_edges(aduanas_port_country, commodity, pform)
        if aduanas_edges:
            export_edges.extend(aduanas_edges)
            _comtrade_covered.add(commodity)
            continue

    # Fall back to Comtrade country-level data distributed by port shares
    dest_data = comtrade_dests.get(commodity, {})
    if dest_data:
        shares = PORT_PRODUCT_MAP.get(port_key, FALLBACK_SHARES.get(port_key, {}))
        new = build_export_edges(dest_data, commodity, pform, "$USD", shares)
        export_edges.extend(new)
        _comtrade_covered.add(commodity)

# Step 3: Any remaining Aduanas-only commodities
# Catch anything in Aduanas that is not in COCHILCO or Comtrade configs
if aduanas_loaded and aduanas_port_country is not None:
    aduanas_commodities = set(aduanas_port_country["COMMODITY"].unique())
    covered = {c[0] for c in COCHILCO_CONFIGS} | _comtrade_covered
    for comm in aduanas_commodities - covered:
        pforms = aduanas_port_country[aduanas_port_country["COMMODITY"] == comm]["PRODUCT_FORM"].unique()
        for pf in pforms:
            apc_edges = build_aduanas_edges(aduanas_port_country, comm, pf)
            if apc_edges:
                export_edges.extend(apc_edges)

# Summary
print("\nExport edge sources:")
_src_counts = {}
for commodity, pform, dest_data, unit, port_key in COCHILCO_CONFIGS:
    _src_counts[commodity] = "COCHILCO/Aduanas"
for commodity, pform, port_key in COMTRADE_MINERALS:
    if commodity in _comtrade_covered:
        _src_counts.setdefault(commodity, "Comtrade/Aduanas")
for comm, src_label in sorted(_src_counts.items()):
    print(f"  {comm:<20} {src_label}")

export_df = pd.DataFrame(export_edges)

# Export edges carry mixed units (kMT, MT, $M_FOB, $USD, $FOB).
# VALUE_USD normalises them to a common basis for downstream maps and aggregations.
_prices_mt = commodity_prices["prices_mt"]
_prices_kg = commodity_prices.get("prices_kg", {})

def _to_usd(row):
    val = row["EXPORT_VALUE"]
    unit = row.get("EXPORT_UNIT", "$USD")
    comm = row["COMMODITIES"]
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return 0.0
    val = float(val)
    if unit in ("$USD", "$FOB"):
        return val
    elif unit == "$M_FOB":
        return val * 1e6
    elif unit == "kMT":
        return val * 1000 * _prices_mt.get(comm, 0)
    elif unit == "MT":
        return val * _prices_mt.get(comm, 0)
    elif unit == "kg":
        return val * _prices_kg.get(comm, 0)
    return val

export_df["VALUE_USD"] = export_df.apply(_to_usd, axis=1)

print(f"\nExport values normalized to USD:")
for comm in export_df["COMMODITIES"].unique():
    sub = export_df[export_df["COMMODITIES"] == comm]
    raw_unit = sub["EXPORT_UNIT"].iloc[0] if len(sub) > 0 else "?"
    print(f"  {comm:<20} {raw_unit:<8} -> ${sub['VALUE_USD'].sum()/1e6:>10,.1f}M USD")

print(f"\nExport edges: {len(export_df)}")
for commodity in export_df["COMMODITIES"].unique():
    sub = export_df[export_df["COMMODITIES"] == commodity]
    print(f"  {commodity:<15} {len(sub):>5} edges ({sub['TO_NAME'].nunique()} countries, {sub['FROM_NAME'].nunique()} ports)")

# Append to unified edge table
edges = edges[edges["EDGE_TYPE"] != "port_to_country"]
export_aligned = export_df[common_cols].copy()
for col in common_cols:
    if col not in export_aligned.columns:
        export_aligned[col] = ""
# Drop all-NA columns before concat to avoid FutureWarning on dtype inference
edges = edges.dropna(axis=1, how="all")
export_aligned = export_aligned.dropna(axis=1, how="all")
edges = pd.concat([edges, export_aligned], ignore_index=True)

print(f"\nUnified edges: {len(edges)}")
for et, count in edges["EDGE_TYPE"].value_counts().items():
    print(f"  {et:<25} {count:>5}")


4. EXPORT DESTINATIONS
Parsing export destination tables...

  Cu refined            26 destinations, total:    1,892.3 kMT
  Cu blister            14 destinations, total:      234.3 kMT
  Cu concentrate        24 destinations, total:    3,725.8 kMT
  Mo concentrate         9 destinations, total:   11,165.9 MT
  Lithium               10 destinations, total:    2,626.6 $M FOB
  Iodine                24 destinations, total:    1,449.7 $M FOB

4A. ADUANAS PORT SHARES (salidas_2024.csv)
  Detected delimiter: semicolon
  Salidas loaded: 317,520 rows x 19 cols
  Export filter: 317,520 -> 294,467 rows
  Mineral rows: 3,848 / 294,467 (1.3%)
  Total mineral FOB: $59,371,857,109

  By commodity:
    Boron               28 rows  $      9,515,274
    Copper            1626 rows  $ 49,485,717,809
    Gold               101 rows  $  1,433,021,397
    Iodine             372 rows  $  1,440,747,267
    Iron                81 rows  $  1,554,456,044
    Lithium            501 rows  $  2,876,973,867
    

/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_93330/3816862029.py:559: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ct_imp_yr["hs4"] = ct_imp_yr["cmdCode"].astype(str).str[:4]
/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_93330/3816862029.py:564: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ct_imp_yr["_mineral"] = ct_imp_yr["hs4"].map(_IMP_HS_MAP)


## 5: Non-Copper Mineral Edges

Domestic supply chain edges for iron, lithium, molybdenum, gold/silver, iodine, nitrate, zinc, rhenium, boron, and sulfuric acid.

In [4]:
# Notebook-specific helpers

def make_edge(from_name, from_type, from_lat, from_lon,
              to_name, to_type, to_lat, to_lon,
              edge_type, product_form, commodity, dist=None):
    """Build a single edge dict matching common_cols."""
    if dist is None and all(pd.notna(x) for x in [from_lat, from_lon, to_lat, to_lon]):
        dist = round(haversine_km(from_lat, from_lon, to_lat, to_lon), 1)
    return {
        "FROM_NAME": from_name, "FROM_TYPE": from_type,
        "FROM_LAT": from_lat,   "FROM_LON": from_lon,
        "TO_NAME": to_name,     "TO_TYPE": to_type,
        "TO_LAT": to_lat,       "TO_LON": to_lon,
        "EDGE_TYPE": edge_type, "PRODUCT_FORM": product_form,
        "COMMODITIES": commodity, "DISTANCE_KM": dist,
    }

def find_facilities(patterns, exclude_mines=False, require_commodity=None):
    """Search inventory by name patterns. Returns matching rows."""
    mask = pd.Series(False, index=inv.index)
    for pat in patterns:
        mask |= inv["FACILITY_NAME"].str.contains(pat, case=False, na=False, regex=False)
    if exclude_mines:
        mask &= ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)
    if require_commodity:
        mask &= inv[comm_col].str.contains(require_commodity, case=False, na=False)
    return inv[mask & inv["LATITUD"].notna()]

def nearest_from_list(lat, lon, port_list):
    """Find nearest port from an explicit list of port names."""
    best_port, best_dist = None, float('inf')
    for p in PORTS:
        if p["name"] in port_list:
            d = haversine_km(lat, lon, p["lat"], p["lon"])
            if d < best_dist:
                best_dist, best_port = d, p
    return best_port, best_dist

# Port coordinate lookup
PORT_DICT = {p['name']: p for p in PORTS}

# Air freight ports (Au/Ag)
AIR_PORTS = {
    "Santiago (air)":    {"lat": -33.39, "lon": -70.79},
    "Antofagasta (air)": {"lat": -23.44, "lon": -70.44},
}

new_edges    = []   # accumulate all non-Cu edges
summary      = {}   # {mineral: {mines: n, edges: n, ...}}
edges_before = len(edges)
print(f"Starting with {edges_before} existing edges")


Starting with 2216 existing edges


In [5]:
# IRON: mine -> pellet plant / processing -> port
# Ports (Aduanas): Caldera 48%, Huasco 39%, Guayacan 12.5%
#
# A second pass after the mine loop catches processing plants that have
# upstream edges but no port connection (e.g. El Romeral, Huasco pellet plant).

section_header("IRON")

# Iron port pool: standard PORTS entries for Caldera/Coquimbo PLUS the
# iron-specific terminals Huasco and Guayacán from pipeline_constants.
# nearest_from_list() only searches the global PORTS list and would
# silently skip Huasco/Guayacán — so we build a merged pool here instead.
_fe_standard  = [p for p in PORTS if p["name"] in {"Caldera", "Coquimbo"}]
_fe_extra     = [
    {"name": nm, "lat": c["lat"], "lon": c["lon"],
     "products": c["products"], "key_users": c["key_users"]}
    for nm, c in FE_PORTS_EXTRA.items()
]
_fe_port_pool = _fe_standard + _fe_extra

def _nearest_fe_port(lat, lon):
    """Return (port_dict, dist_km) from the iron-specific port pool.
    Includes Huasco and Guayacán which are absent from the main PORTS list.
    """
    best, best_d = None, float("inf")
    for p in _fe_port_pool:
        d = haversine_km(lat, lon, p["lat"], p["lon"])
        if d < best_d:
            best_d, best = d, p
    return best, best_d

fe_mines = inv[
    (inv.get("COCHILCO_FE_2024_KMT", pd.Series(dtype=float)).notna()) &
    (inv["COCHILCO_FE_2024_KMT"] > 0) &
    inv["LATITUD"].notna()
].copy()

# Find iron processing (pellet plants, grinding plants)
fe_plants = find_facilities(
    ["pellet", "hierro", "iron", "CMP", "CAP", "grinding", "Huasco"],
    exclude_mines=True
)
# Also check for facilities with Iron in commodity list
fe_plants_comm = inv[
    inv[comm_col].str.contains("Iron", case=False, na=False) &
    ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False) &
    inv["LATITUD"].notna()
]
fe_plants = pd.concat([fe_plants, fe_plants_comm]).drop_duplicates(subset="FACILITY_NAME")

print(f"  Iron mines with production: {len(fe_mines)}")
for _, m in fe_mines.iterrows():
    print(f"    {m['FACILITY_NAME']:<40} {m['COCHILCO_FE_2024_KMT']:>8.1f} kMT")
print(f"  Iron processing facilities found: {len(fe_plants)}")
for _, p in fe_plants.iterrows():
    print(f"    {p['FACILITY_NAME']:<50} {p['FACILITY_TYPE']}")

fe_edge_count = 0
fe_plants_with_port = set()  # track which plants got a port edge

for _, mine in fe_mines.iterrows():
    mlat, mlon = mine["LATITUD"], mine["LONGITUD"]
    mine_name = mine["FACILITY_NAME"]

    # Try to find a nearby processing plant (<100 km)
    linked_to_plant = False
    if len(fe_plants) > 0:
        fe_plants_copy = fe_plants.copy()
        fe_plants_copy["_dist"] = fe_plants_copy.apply(
            lambda r: haversine_km(mlat, mlon, r["LATITUD"], r["LONGITUD"]), axis=1)
        nearby = fe_plants_copy[fe_plants_copy["_dist"] < 100].sort_values("_dist")
        if len(nearby) > 0:
            plant = nearby.iloc[0]
            new_edges.append(make_edge(
                mine_name, "mine", mlat, mlon,
                plant["FACILITY_NAME"], "plant", plant["LATITUD"], plant["LONGITUD"],
                "mine_to_plant", "iron_ore", "Iron"))
            fe_edge_count += 1
            linked_to_plant = True
            # Plant to port
            port, dist = _nearest_fe_port(plant["LATITUD"], plant["LONGITUD"])
            if port:
                # Check if plant-to-port edge already exists
                existing = edges[
                    (edges["FROM_NAME"] == plant["FACILITY_NAME"]) &
                    (edges["TO_NAME"] == port["name"]) &
                    (edges["COMMODITIES"].str.contains("Iron", case=False, na=False))
                ]
                if len(existing) == 0:
                    new_edges.append(make_edge(
                        plant["FACILITY_NAME"], "plant",
                        plant["LATITUD"], plant["LONGITUD"],
                        port["name"], "port", port["lat"], port["lon"],
                        "iron_to_port", "iron_ore", "Iron"))
                    fe_edge_count += 1
                fe_plants_with_port.add(plant["FACILITY_NAME"])
            print(f"    {mine_name:<35} -> {plant['FACILITY_NAME']:<30} -> {port['name'] if port else 'NO PORT'}")

    # Direct mine-to-port if no plant found
    if not linked_to_plant:
        port = None
        best_dist = float("inf")
        # Use iron port pool (Caldera, Coquimbo, Huasco, Guayacán)
        port, best_dist = _nearest_fe_port(mlat, mlon)
        if port:
            new_edges.append(make_edge(
                mine_name, "mine", mlat, mlon,
                port["name"], "port", port["lat"], port["lon"],
                "iron_to_port", "iron_ore", "Iron"))
            fe_edge_count += 1
            print(f"    {mine_name:<35} -> (direct) -> {port['name']}")

# Some pellet/processing plants have upstream edges from USGS/SERNAGEOMIN
# linkage data but were not matched in the mine loop above.
# Sweep for these dead ends and add port edges.
print(f"\n  Checking for iron plant dead ends...")
fe_plant_names = set(fe_plants["FACILITY_NAME"].values)
for plant_name in fe_plant_names:
    if plant_name in fe_plants_with_port:
        continue
    # Check if this plant has upstream edges
    upstream = edges[edges["TO_NAME"] == plant_name]
    # Also check new_edges
    upstream_new = [e for e in new_edges if e["TO_NAME"] == plant_name]
    if len(upstream) == 0 and len(upstream_new) == 0:
        continue
    # This plant has upstream but no port edge from us: add one
    plant_row = fe_plants[fe_plants["FACILITY_NAME"] == plant_name].iloc[0]
    plat, plon = plant_row["LATITUD"], plant_row["LONGITUD"]
    port, dist = _nearest_fe_port(plat, plon)
    if port:
        new_edges.append(make_edge(
            plant_name, "plant", plat, plon,
            port["name"], "port", port["lat"], port["lon"],
            "iron_to_port", "iron_ore", "Iron"))
        fe_edge_count += 1
        fe_plants_with_port.add(plant_name)
        print(f"    (dead-end fix) {plant_name:<35} -> {port['name']} ({dist:.0f} km)")

summary["Iron"] = {"mines": len(fe_mines), "edges": fe_edge_count}
print(f"\n  Iron edges added: {fe_edge_count}")



IRON
  Iron mines with production: 10
    Boqueron Chañar                               7.3 kMT
    Cerro Negro Norte                          1898.2 kMT
    Cristales                                  2196.3 kMT
    Distrito El Algarrobo                      3449.5 kMT
    Dominga                                     850.1 kMT
    El Algarrobo                                644.3 kMT
    Los Colorados                              2928.8 kMT
    Romeral                                     146.3 kMT
    Santo Domingo                              6717.2 kMT
    Tofo                                        703.8 kMT
  Iron processing facilities found: 16
    Coronel grinding plant                             Grinding Plant
    El Romeral pellet-feed plant                       Pellet Plant
    Grinding plant at Puerto Montt                     Grinding Plant
    Grinding plant at San Antonio                      Grinding Plant
    Huasco concentration plant                         Concentra

In [6]:
# LITHIUM: plant -> port (mine-plant links already exist from Part 2D)
# Ports (Aduanas): Angamos 93.3%
# Production: parse from COCHILCO non-metallic tables if available

section_header("LITHIUM")

LI_PORTS = ["Angamos", "Mejillones", "Antofagasta (ATI)"]

# Find lithium processing plants (already in inventory from Part 2D)
li_plants = inv[
    inv[comm_col].str.contains("Lithium", case=False, na=False) &
    ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False) &
    inv["LATITUD"].notna()
].copy()

# Also find lithium mines
li_mines = inv[
    inv[comm_col].str.contains("Lithium", case=False, na=False) &
    inv["FACILITY_TYPE"].str.contains("Mine.*active", case=True, na=False, regex=True) &
    inv["LATITUD"].notna()
].copy()

print(f"  Lithium mines: {len(li_mines)}")
for _, m in li_mines.iterrows():
    print(f"    {m['FACILITY_NAME']:<50} {m['FACILITY_TYPE']}")
print(f"  Lithium processing plants: {len(li_plants)}")
for _, p in li_plants.iterrows():
    print(f"    {p['FACILITY_NAME']:<50} {p['FACILITY_TYPE']}")

# Build plant-to-port edges for each Li processing plant
li_edge_count = 0

# Deduplicate by location (Part 2D already deduped, but be safe)
if len(li_plants) > 0:
    li_plants_dedup = li_plants.copy()
    li_plants_dedup["_loc"] = (
        li_plants_dedup["LATITUD"].round(2).astype(str) + "_" +
        li_plants_dedup["LONGITUD"].round(2).astype(str))
    li_plants_dedup = li_plants_dedup.drop_duplicates(subset=["_loc"], keep="first")

    for _, plant in li_plants_dedup.iterrows():
        port, dist = nearest_from_list(plant["LATITUD"], plant["LONGITUD"], LI_PORTS)
        if port:
            new_edges.append(make_edge(
                plant["FACILITY_NAME"], "plant",
                plant["LATITUD"], plant["LONGITUD"],
                port["name"], "port", port["lat"], port["lon"],
                "lithium_to_port", "lithium_compounds", "Lithium"))
            li_edge_count += 1
            print(f"    {plant['FACILITY_NAME']:<45} -> {port['name']} ({dist:.0f} km)")

# If no Li plants found, connect mines directly to port
if li_edge_count == 0 and len(li_mines) > 0:
    print("  No Li plants with coords; connecting mines directly to port")
    for _, mine in li_mines.iterrows():
        port, dist = nearest_from_list(mine["LATITUD"], mine["LONGITUD"], LI_PORTS)
        if port:
            new_edges.append(make_edge(
                mine["FACILITY_NAME"], "mine",
                mine["LATITUD"], mine["LONGITUD"],
                port["name"], "port", port["lat"], port["lon"],
                "lithium_to_port", "lithium_compounds", "Lithium"))
            li_edge_count += 1

summary["Lithium"] = {"mines": len(li_mines), "plants": len(li_plants), "edges": li_edge_count}
print(f"\n  Lithium edges added: {li_edge_count}")


LITHIUM
  Lithium mines: 1
    Salar de Atacama                                   Mine (active)
  Lithium processing plants: 9
    Chemetall Foote plant                              Processing Plant
    Chemetall Foote plant                              Processing Plant
    Chemetall Foote plant                              Processing Plant
    Dual-use plant at Salar del Carmen                 Processing Plant
    Mine brines of Atacama Salar                       Processing Plant
    Plant at Salar del Carmen                          Processing Plant
    Plant at Salar del Carmen                          Processing Plant
    Salar de Atacama                                   Processing Plant
    Three KCl plants at Salar del Carmen               Processing Plant
    Chemetall Foote plant                         -> Antofagasta (ATI) (15 km)
    Dual-use plant at Salar del Carmen            -> Antofagasta (ATI) (14 km)
    Mine brines of Atacama Salar                  -> Antofagasta (

In [7]:
# MOLYBDENUM: mine -> Mo plant -> port
# Mo is separated from Cu concentrate at dedicated Mo plants.
# Key plants: Molynor (Mejillones), Molymet (Nos/San Bernardo)
# Ports (Aduanas): Angamos 45%, San Antonio 26%, ATI 15%

section_header("MOLYBDENUM")

MO_PORTS = ["Angamos", "Mejillones", "San Antonio", "Antofagasta (ATI)"]

# Find Mo production mines
mo_mines = inv[
    (inv.get("COCHILCO_MO_2024_MT", pd.Series(dtype=float)).notna()) &
    (inv["COCHILCO_MO_2024_MT"] > 0) &
    inv["LATITUD"].notna()
].copy()

# Find Mo processing plants
mo_patterns = ["molib", "molyb", "molynor", "molymet", "molibdeno"]
mo_plants = find_facilities(mo_patterns, exclude_mines=True)
# Also search by commodity
mo_plants_comm = inv[
    inv[comm_col].str.contains("Molybdenum", case=False, na=False) &
    ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False) &
    inv["LATITUD"].notna()
]
mo_plants = pd.concat([mo_plants, mo_plants_comm]).drop_duplicates(subset="FACILITY_NAME")

print(f"  Mo mines with production: {len(mo_mines)}")
for _, m in mo_mines.iterrows():
    print(f"    {m['FACILITY_NAME']:<40} {m['COCHILCO_MO_2024_MT']:>8.1f} MT")
print(f"  Mo processing facilities: {len(mo_plants)}")
for _, p in mo_plants.iterrows():
    print(f"    {p['FACILITY_NAME']:<50} {p['FACILITY_TYPE']}")

mo_edge_count = 0
mo_plants_linked = set()  # track which plants got mine-to-plant edges

for _, mine in mo_mines.iterrows():
    mlat, mlon = mine["LATITUD"], mine["LONGITUD"]
    mine_name = mine["FACILITY_NAME"]

    if len(mo_plants) > 0:
        # Assign to nearest Mo plant (Mo concentrate is trucked/railed, <500km typical)
        mo_plants_c = mo_plants.copy()
        mo_plants_c["_dist"] = mo_plants_c.apply(
            lambda r: haversine_km(mlat, mlon, r["LATITUD"], r["LONGITUD"]), axis=1)
        nearest = mo_plants_c.sort_values("_dist").iloc[0]
        new_edges.append(make_edge(
            mine_name, "mine", mlat, mlon,
            nearest["FACILITY_NAME"], "mo_plant",
            nearest["LATITUD"], nearest["LONGITUD"],
            "mine_to_mo_plant", "mo_concentrate", "Molybdenum"))
        mo_edge_count += 1
        mo_plants_linked.add(nearest["FACILITY_NAME"])
        print(f"    {mine_name:<35} -> {nearest['FACILITY_NAME']:<30} ({nearest['_dist']:.0f} km)")
    else:
        # No Mo plant found; direct to port
        port, dist = nearest_from_list(mlat, mlon, MO_PORTS)
        if port:
            new_edges.append(make_edge(
                mine_name, "mine", mlat, mlon,
                port["name"], "port", port["lat"], port["lon"],
                "mo_to_port", "mo_concentrate", "Molybdenum"))
            mo_edge_count += 1

# Mo plant -> port edges
for plant_name in mo_plants_linked:
    plant_row = mo_plants[mo_plants["FACILITY_NAME"] == plant_name].iloc[0]
    port, dist = nearest_from_list(plant_row["LATITUD"], plant_row["LONGITUD"], MO_PORTS)
    if port:
        new_edges.append(make_edge(
            plant_name, "mo_plant",
            plant_row["LATITUD"], plant_row["LONGITUD"],
            port["name"], "port", port["lat"], port["lon"],
            "mo_to_port", "mo_concentrate", "Molybdenum"))
        mo_edge_count += 1
        print(f"    (plant->port) {plant_name:<35} -> {port['name']}")

summary["Molybdenum"] = {"mines": len(mo_mines), "plants": len(mo_plants), "edges": mo_edge_count}
print(f"\n  Mo edges added: {mo_edge_count}")


MOLYBDENUM
  Mo mines with production: 12
    Andina                                     1584.7 MT
    Caserones                                  3183.5 MT
    Centinela                                  2407.6 MT
    Chuquicamata                               4067.7 MT
    Collahuasi                                 2054.5 MT
    El Teniente                                4345.4 MT
    Los Bronces                                1637.7 MT
    Los Pelambres                              8402.0 MT
    Quebrada Blanca                             672.0 MT
    Radomiro Tomic                             6101.6 MT
    Sierra Gorda                               2807.7 MT
    Spence                                      637.1 MT
  Mo processing facilities: 4
    Molybdenum plant                                   Processing Plant
    Tortolas molybdenum flotation plant                Concentrator
    El Teniente plant                                  Processing Plant
    Los Pelambres plant        

In [8]:
# GOLD + SILVER: mine -> smelter/refinery -> air port
# Au/Ag are byproducts recovered at Cu smelters, refined, then shipped
# via air freight. Dedicated Au/Ag mines (El Penon, etc.) send ore to
# nearby Cu smelters or own processing.
# Ports (Aduanas): Santiago (air) ~100% for both
#
# Restricted to named SMELTERS entries plus explicit refinery-stage facilities
# to avoid routing Au/Ag to generic processing plants (which have no airport edge).

section_header("GOLD & SILVER")

# Build a curated set of smelter/refinery names for Au/Ag routing.
# Use the canonical inventory names from smelter_inv_map, plus any
# refinery-stage facilities.
_auag_target_names = set(smelter_inv_map.values())
# Add refineries (Biocobre, Gregorio, etc.)
_refinery_mask = (
    (inv["CHAIN_STAGE"].isin(["refining", "smelting"])) &
    inv["LATITUD"].notna() &
    (
        inv["FACILITY_NAME"].str.contains("refin|Gregorio|Biocobre", case=False, na=False, regex=True) |
        inv["FACILITY_NAME"].isin(_auag_target_names)
    )
)
refinery_rows = inv[_refinery_mask].copy()
# Deduplicate: if the same name appears multiple times (commodity variants),
# keep only the first to avoid routing Au/Ag to dead-end duplicates.
refinery_rows = refinery_rows.drop_duplicates(subset="FACILITY_NAME", keep="first")

print(f"  Au/Ag target smelters/refineries: {len(refinery_rows)}")
for _, r in refinery_rows.iterrows():
    print(f"    {r['FACILITY_NAME'][:55]:<55} {r['FACILITY_TYPE']}")

for mineral, col_name, unit in [
    ("Gold", "COCHILCO_AU_2024_KG", "Kg"),
    ("Silver", "COCHILCO_AG_2024_KG", "Kg"),
]:
    print(f"\n  --- {mineral} ---")

    if col_name not in inv.columns:
        print(f"  Column {col_name} not found, skipping")
        summary[mineral] = {"mines": 0, "edges": 0}
        continue

    pm_mines = inv[
        (inv[col_name].notna()) & (inv[col_name] > 0) & inv["LATITUD"].notna()
    ].copy()

    print(f"  Mines with {mineral} production: {len(pm_mines)}")

    pm_edge_count = 0
    product_form = "gold_refined" if mineral == "Gold" else "silver_refined"

    for _, mine in pm_mines.iterrows():
        mlat, mlon = mine["LATITUD"], mine["LONGITUD"]
        mine_name = mine["FACILITY_NAME"]
        prod = mine[col_name]

        # Find nearest smelter/refinery (Au/Ag is processed at Cu smelters)
        if len(refinery_rows) > 0:
            dists = refinery_rows.apply(
                lambda r: haversine_km(mlat, mlon, r["LATITUD"], r["LONGITUD"]), axis=1)
            nearest_idx = dists.idxmin()
            nearest_ref = refinery_rows.loc[nearest_idx]
            ref_dist = dists.loc[nearest_idx]

            # Mine -> smelter/refinery
            new_edges.append(make_edge(
                mine_name, "mine", mlat, mlon,
                nearest_ref["FACILITY_NAME"], "smelter",
                nearest_ref["LATITUD"], nearest_ref["LONGITUD"],
                "mine_to_smelter", product_form, mineral))
            pm_edge_count += 1

    # Smelter/refinery -> air port edges (deduplicated)
    # All refined Au/Ag goes to Santiago (air), with minor Antofagasta (air)
    smelters_used = set()
    for e in new_edges:
        if e["COMMODITIES"] == mineral and e["EDGE_TYPE"] == "mine_to_smelter":
            smelters_used.add(e["TO_NAME"])

    for smelter_name in smelters_used:
        sr = refinery_rows[refinery_rows["FACILITY_NAME"] == smelter_name]
        if len(sr) == 0:
            continue
        sr = sr.iloc[0]
        # Assign to nearest air port
        best_air, best_d = None, float("inf")
        for aname, acoords in AIR_PORTS.items():
            d = haversine_km(sr["LATITUD"], sr["LONGITUD"], acoords["lat"], acoords["lon"])
            if d < best_d:
                best_d, best_air = d, (aname, acoords)
        if best_air:
            aname, acoords = best_air
            new_edges.append(make_edge(
                smelter_name, "smelter", sr["LATITUD"], sr["LONGITUD"],
                aname, "port", acoords["lat"], acoords["lon"],
                f"{mineral.lower()}_to_airport", product_form, mineral))
            pm_edge_count += 1
            print(f"    (smelter->air) {smelter_name[:45]:<45} -> {aname}")

    summary[mineral] = {"mines": len(pm_mines), "edges": pm_edge_count}
    print(f"  {mineral} edges added: {pm_edge_count}")



GOLD & SILVER
  Au/Ag target smelters/refineries: 13
    Aconcagua refineries                                    Refinery
    Altonorte smelter                                       Smelter
    Bio Bio refineries                                      Refinery
    Biocobre refinery                                       Refinery
    Caletones smelter (anodes). refinery (fire-refined ingo Smelter
    Chagres smelter                                         Smelter
    Chuquicamata SX-EW plant (oxide) and smelter            Smelter
    El Teniente plant                                       Processing Plant
    Gregorio refineries                                     Refinery
    Hernán Videla Lira smelter                              Smelter
    Las Ventanas refinery and smelter                       Smelter
    Potrerillos SX-EW refinery and smelter                  Smelter
    Ventanas refinery and smelter                           Smelter

  --- Gold ---
  Mines with Gold production: 52


In [9]:
# IODINE: identify facilities -> plant -> port
# Chile is world #1 producer. Main producers:
#   SQM (Nueva Victoria, Pedro de Valdivia, Iris, Maria Elena)
#   Algorta Norte (ACF) in Tarapaca
#   Cosayach in Tarapaca
# Ports (Aduanas): Angamos 46%, Iquique 22%, Valparaiso 17%
#
# Dedup is by facility name, not coordinates — rounding coords to 1 decimal
# was silently dropping Maria Elena and Nueva Victoria.

section_header("IODINE")

IO_PORTS = ["Angamos", "Mejillones", "Iquique"]

io_patterns = [
    "iod", "yod", "caliche", "Nueva Victoria", "Pedro de Valdivia",
    "Maria Elena", "Algorta", "Cosayach", "Iris", "Pampa",
    "SQM", "Cala Cala"
]
io_facilities = find_facilities(io_patterns)

# Also search by commodity (catches accent variants like "Maria" vs "María")
io_comm = inv[
    inv[comm_col].str.contains("Iodine", case=False, na=False) &
    inv["LATITUD"].notna()
]
io_all = pd.concat([io_facilities, io_comm]).drop_duplicates(subset="FACILITY_NAME")

io_mines = io_all[io_all["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)]
io_plants = io_all[~io_all["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)]

print(f"  Iodine-related facilities: {len(io_all)} ({len(io_mines)} mines, {len(io_plants)} plants)")
for _, f in io_all.iterrows():
    print(f"    {f['FACILITY_NAME']:<50} {f['FACILITY_TYPE']:<20} {f.get('REGION', '')}")

io_edge_count = 0

# Mine -> plant edges for iodine
for _, mine in io_mines.iterrows():
    if len(io_plants) > 0:
        dists = io_plants.apply(
            lambda r: haversine_km(mine["LATITUD"], mine["LONGITUD"],
                                   r["LATITUD"], r["LONGITUD"]), axis=1)
        nearby = io_plants.loc[dists[dists < 150].index]
        for idx in nearby.index:
            new_edges.append(make_edge(
                mine["FACILITY_NAME"], "mine", mine["LATITUD"], mine["LONGITUD"],
                io_plants.at[idx, "FACILITY_NAME"], "plant",
                io_plants.at[idx, "LATITUD"], io_plants.at[idx, "LONGITUD"],
                "mine_to_plant", "iodine", "Iodine"))
            io_edge_count += 1

# Dedup by FACILITY_NAME — coordinate rounding was silently merging
# distinct plants. This ensures every iodine plant gets a port edge, even if
# two plants share similar coordinates (e.g. Maria Elena plant for
# Nitrate vs Maria Elena plant for Iodine, or Iris Plant vs Nueva
# Victoria plant in nearby Tarapaca locations).
port_sources = io_plants if len(io_plants) > 0 else io_mines
if len(port_sources) > 0:
    port_sources_dedup = port_sources.drop_duplicates(subset=["FACILITY_NAME"], keep="first")

    for _, src in port_sources_dedup.iterrows():
        port, dist = nearest_from_list(src["LATITUD"], src["LONGITUD"], IO_PORTS)
        if port:
            new_edges.append(make_edge(
                src["FACILITY_NAME"], "plant" if src["FACILITY_NAME"] in io_plants["FACILITY_NAME"].values else "mine",
                src["LATITUD"], src["LONGITUD"],
                port["name"], "port", port["lat"], port["lon"],
                "iodine_to_port", "iodine", "Iodine"))
            io_edge_count += 1
            print(f"    {src['FACILITY_NAME']:<45} -> {port['name']} ({dist:.0f} km)")

if io_edge_count == 0:
    print("  WARNING: No iodine facilities found in inventory.")
    print("  Consider adding SQM/ACF iodine plants manually.")

summary["Iodine"] = {"facilities": len(io_all), "edges": io_edge_count}
print(f"\n  Iodine edges added: {io_edge_count}")



IODINE
  Iodine-related facilities: 25 (17 mines, 6 plants)
    Cala Cala                                          Mine (active)        Tarapacá
    Calichera                                          Mine (idle)          Tarapacá
    Pampa Camarones                                    Mine (active)        Arica y Parinacota
    Pampa Escondida                                    Prospect/Project     Antofagasta
    Pampa Paciencia                                    Prospect/Project     Antofagasta
    Pedro de Valdivia                                  Mine (idle)          Antofagasta
    Pampa Blanca                                       Mine (active)        Antofagasta
    Nueva Victoria                                     Mine (active)        Tarapacá
    Pampa Orcoma                                       Mine (active)        Tarapacá
    Iris Plant                                         Processing Plant     Tarapacá
    Maria Elena plant                                  Processing P

In [10]:
# NITRATE: facility -> port
# SQM salar operations in Tarapaca/Antofagasta
# Ports (Aduanas): Tocopilla 84%, Iquique 10%

section_header("NITRATE")

# Tocopilla is not in the standard PORTS list; define coords
# Tocopilla handles ~84% of nitrate FOB but is not in the main PORTS list.

NO3_PORTS = ["Iquique", "Angamos"]

no3_patterns = [
    "nitrat", "salitre", "caliche", "Pedro de Valdivia", "Maria Elena",
    "Coya Sur", "Pampa Blanca", "Cosayach", "SQM", "Tocopilla"
]
no3_facilities = find_facilities(no3_patterns)
no3_comm = inv[
    inv[comm_col].str.contains("Nitrate|Potassium|Potash", case=False, na=False) &
    inv["LATITUD"].notna()
]
no3_all = pd.concat([no3_facilities, no3_comm]).drop_duplicates(subset="FACILITY_NAME")

print(f"  Nitrate-related facilities: {len(no3_all)}")
for _, f in no3_all.iterrows():
    print(f"    {f['FACILITY_NAME']:<50} {f['FACILITY_TYPE']}")

no3_edge_count = 0
# Facility -> port (prefer Tocopilla, then Iquique)
no3_plants = no3_all[
    ~no3_all["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)
]
sources = no3_plants if len(no3_plants) > 0 else no3_all

for _, src in sources.iterrows():
    slat, slon = src["LATITUD"], src["LONGITUD"]
    # Check Tocopilla first
    best_port, best_dist = None, float("inf")
    for pname, pcoords in NITRATE_PORT_COORDS.items():
        d = haversine_km(slat, slon, pcoords["lat"], pcoords["lon"])
        if d < best_dist:
            best_dist, best_port = d, {"name": pname, **pcoords}
    for p in PORTS:
        if p["name"] in NO3_PORTS:
            d = haversine_km(slat, slon, p["lat"], p["lon"])
            if d < best_dist:
                best_dist, best_port = d, p
    if best_port:
        new_edges.append(make_edge(
            src["FACILITY_NAME"], "plant", slat, slon,
            best_port["name"], "port", best_port["lat"], best_port["lon"],
            "nitrate_to_port", "nitrate", "Nitrate"))
        no3_edge_count += 1
        print(f"    {src['FACILITY_NAME']:<45} -> {best_port['name']} ({best_dist:.0f} km)")

summary["Nitrate"] = {"facilities": len(no3_all), "edges": no3_edge_count}
print(f"\n  Nitrate edges added: {no3_edge_count}")


NITRATE
  Nitrate-related facilities: 33
    Calichera                                          Mine (idle)
    Distrito Tocopilla                                 Mine (idle)
    Pedro de Valdivia                                  Mine (idle)
    Pampa Blanca                                       Mine (active)
    Coya Sur plant                                     Processing Plant
    Maria Elena plant                                  Processing Plant
    Plant near Tocopilla                               Processing Plant
    Aguas Blancas                                      Mine (idle)
    Cala Cala                                          Mine (active)
    Chiquinquira                                       Mine (idle)
    Florencia                                          Mine (idle)
    Lagunas                                            Mine (idle)
    María Elena                                        Mine (active)
    Negreiros                                          Mine (activ

In [11]:
# ZINC: mine -> port
# El Toqui (Aysen) + Alhue (Santiago)
# No Aduanas exports captured under HS 2608; may be domestic or
# exported under different classification. Build chain anyway.

section_header("ZINC")

zn_mines = inv[
    (inv.get("COCHILCO_ZN_2024_MT", pd.Series(dtype=float)).notna()) &
    (inv["COCHILCO_ZN_2024_MT"] > 0) &
    inv["LATITUD"].notna()
].copy()

# Also search for zinc processing
zn_plants = find_facilities(["zinc", "toqui"], exclude_mines=True)
zn_plants_comm = inv[
    inv[comm_col].str.contains("Zinc", case=False, na=False) &
    ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False) &
    inv["LATITUD"].notna()
]
zn_plants = pd.concat([zn_plants, zn_plants_comm]).drop_duplicates(subset="FACILITY_NAME")

# Puerto Aysen / Chacabuco for El Toqui; San Antonio for Alhue
# Puerto Aysén serves El Toqui mine (Region Aysén) and is not in the main PORTS list.

print(f"  Zinc mines with production: {len(zn_mines)}")
for _, m in zn_mines.iterrows():
    print(f"    {m['FACILITY_NAME']:<40} {m['COCHILCO_ZN_2024_MT']:>8.1f} MT")
print(f"  Zinc processing: {len(zn_plants)}")

zn_edge_count = 0
for _, mine in zn_mines.iterrows():
    mlat, mlon = mine["LATITUD"], mine["LONGITUD"]

    # Link to plant if any
    if len(zn_plants) > 0:
        dists = zn_plants.apply(
            lambda r: haversine_km(mlat, mlon, r["LATITUD"], r["LONGITUD"]), axis=1)
        nearby = zn_plants.loc[dists[dists < 100].index]
        for idx in nearby.index:
            new_edges.append(make_edge(
                mine["FACILITY_NAME"], "mine", mlat, mlon,
                zn_plants.at[idx, "FACILITY_NAME"], "plant",
                zn_plants.at[idx, "LATITUD"], zn_plants.at[idx, "LONGITUD"],
                "mine_to_plant", "zinc_concentrate", "Zinc"))
            zn_edge_count += 1

    # Mine (or plant) -> port
    best_port, best_dist = None, float("inf")
    for p in PORTS:
        d = haversine_km(mlat, mlon, p["lat"], p["lon"])
        if d < best_dist:
            best_dist, best_port = d, p
    for pname, pcoords in ZN_PORT_COORDS.items():
        d = haversine_km(mlat, mlon, pcoords["lat"], pcoords["lon"])
        if d < best_dist:
            best_dist, best_port = d, {"name": pname, **pcoords}
    if best_port:
        new_edges.append(make_edge(
            mine["FACILITY_NAME"], "mine", mlat, mlon,
            best_port["name"], "port", best_port["lat"], best_port["lon"],
            "zinc_to_port", "zinc_concentrate", "Zinc"))
        zn_edge_count += 1
        print(f"    {mine['FACILITY_NAME']:<40} -> {best_port['name']} ({best_dist:.0f} km)")

summary["Zinc"] = {"mines": len(zn_mines), "edges": zn_edge_count}
print(f"\n  Zinc edges added: {zn_edge_count}")


ZINC
  Zinc mines with production: 2
    Alhué                                      6060.0 MT
    El Toqui                                  25899.0 MT
  Zinc processing: 2
    Alhué                                    -> San Antonio (75 km)
    El Toqui                                 -> Puerto Aysen (76 km)

  Zinc edges added: 4


In [12]:
# RHENIUM: Mo plant -> Re refinery -> port
# Byproduct of Mo roasting. Molymet (Nos, San Bernardo) is the
# world's largest Re producer.
# Ports (Aduanas): San Antonio 97%, Valparaiso 3%
#
# Filter to processing plants and refineries only — copper mines tagged
# with Rhenium in their commodity list would otherwise generate false
# Mo-plant-to-copper-mine edges.

section_header("RHENIUM")

RE_PORTS = ["San Antonio", "Ventanas"]

# Find Molymet / rhenium facilities
re_patterns = ["molymet", "rhenium", "renio", "perrhenate"]
re_facilities = find_facilities(re_patterns)
re_comm = inv[
    inv[comm_col].str.contains("Rhenium", case=False, na=False) &
    inv["LATITUD"].notna()
]
re_all_raw = pd.concat([re_facilities, re_comm]).drop_duplicates(subset="FACILITY_NAME")

print(f"  Rhenium-related facilities (raw): {len(re_all_raw)}")
for _, f in re_all_raw.iterrows():
    print(f"    {f['FACILITY_NAME']:<50} {f['FACILITY_TYPE']}")

# Filter to processing plants/refineries — mines and prospects are upstream
# of Mo plants, not downstream Re targets.
re_plants = re_all_raw[
    ~re_all_raw["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)
].copy()

print(f"\n  Re processing plants (filtered): {len(re_plants)}")
for _, f in re_plants.iterrows():
    print(f"    {f['FACILITY_NAME']:<50} {f['FACILITY_TYPE']}")

re_edge_count = 0

# Mo plant -> Re plant edges (only actual Re processing facilities)
if len(re_plants) > 0 and len(mo_plants) > 0:
    for _, mo_p in mo_plants.iterrows():
        for _, re_f in re_plants.iterrows():
            if mo_p["FACILITY_NAME"] != re_f["FACILITY_NAME"]:
                new_edges.append(make_edge(
                    mo_p["FACILITY_NAME"], "mo_plant",
                    mo_p["LATITUD"], mo_p["LONGITUD"],
                    re_f["FACILITY_NAME"], "re_plant",
                    re_f["LATITUD"], re_f["LONGITUD"],
                    "mo_to_re_plant", "perrhenate", "Rhenium"))
                re_edge_count += 1
                print(f"    (mo->re) {mo_p['FACILITY_NAME']:<35} -> {re_f['FACILITY_NAME']}")

# Re plant -> port (only actual Re plants, not mines)
for _, src in re_plants.iterrows():
    port, dist = nearest_from_list(src["LATITUD"], src["LONGITUD"], RE_PORTS)
    if port:
        new_edges.append(make_edge(
            src["FACILITY_NAME"], "re_plant",
            src["LATITUD"], src["LONGITUD"],
            port["name"], "port", port["lat"], port["lon"],
            "rhenium_to_port", "perrhenate", "Rhenium"))
        re_edge_count += 1
        print(f"    (re->port) {src['FACILITY_NAME']:<40} -> {port['name']} ({dist:.0f} km)")

if re_edge_count == 0:
    print("  WARNING: No rhenium processing facilities found.")
    print("  Molymet (Nos, Santiago) may not be in inventory.")

summary["Rhenium"] = {"plants": len(re_plants), "edges": re_edge_count}
print(f"\n  Rhenium edges added: {re_edge_count}")



RHENIUM
  Rhenium-related facilities (raw): 14
    Andina                                             Mine (active)
    Caserones                                          Mine (active)
    Centinela                                          Mine (active)
    Chuquicamata                                       Mine (active)
    Collahuasi                                         Mine (active)
    El Teniente                                        Mine (active)
    Los Bronces                                        Mine (active)
    Los Bronces Subterráneo                            Prospect/Project
    Los Bronces Sur                                    Prospect/Project
    Los Pelambres                                      Mine (active)
    Radomiro Tomic                                     Mine (active)
    Salvador                                           Mine (active)
    Sierra Gorda                                       Mine (active)
    Nos plant                                    

In [13]:
# BORON: facility -> port
# Small borate operations in Arica/Antofagasta
# Ports (Aduanas): ATI 69%, Arica 23%
#
# Filter to active mines and processing plants — prospects and far-field
# facilities (e.g. a Santiago plant 1,083 km from the nearest port)
# would otherwise generate implausible edges.

section_header("BORON")

BORON_PORTS = ["Antofagasta (ATI)", "Iquique"]
# Arica handles ~23% of boron FOB but is not in the main PORTS list.

boron_patterns = ["borat", "boro", "ulexita", "boron"]
boron_facilities = find_facilities(boron_patterns)
boron_comm = inv[
    inv[comm_col].str.contains("Boron|Borate", case=False, na=False) &
    inv["LATITUD"].notna()
]
boron_all_raw = pd.concat([boron_facilities, boron_comm]).drop_duplicates(subset="FACILITY_NAME")

print(f"  Boron facilities (raw): {len(boron_all_raw)}")

# Filter to active mines and processing plants.
boron_all = boron_all_raw[
    boron_all_raw["FACILITY_TYPE"].str.contains("Mine.*active|Processing|Concentrator", case=False, na=False, regex=True)
].copy()

# Also cap distance: exclude facilities > 800 km from any boron port
# (catches the Santiago Metropolitan Region plant)
def _min_boron_port_dist(row):
    d_min = float("inf")
    for p in PORTS:
        if p["name"] in BORON_PORTS:
            d = haversine_km(row["LATITUD"], row["LONGITUD"], p["lat"], p["lon"])
            d_min = min(d_min, d)
    for pname, pcoords in BORON_PORT_COORDS.items():
        d = haversine_km(row["LATITUD"], row["LONGITUD"], pcoords["lat"], pcoords["lon"])
        d_min = min(d_min, d)
    return d_min

boron_all = boron_all[boron_all.apply(_min_boron_port_dist, axis=1) < 800]

print(f"  Boron facilities (filtered): {len(boron_all)}")
for _, f in boron_all.iterrows():
    print(f"    {f['FACILITY_NAME']:<50} {f['FACILITY_TYPE']}")

boron_edge_count = 0
for _, src in boron_all.iterrows():
    slat, slon = src["LATITUD"], src["LONGITUD"]
    best_port, best_dist = None, float("inf")
    for p in PORTS:
        if p["name"] in BORON_PORTS:
            d = haversine_km(slat, slon, p["lat"], p["lon"])
            if d < best_dist:
                best_dist, best_port = d, p
    for pname, pcoords in BORON_PORT_COORDS.items():
        d = haversine_km(slat, slon, pcoords["lat"], pcoords["lon"])
        if d < best_dist:
            best_dist, best_port = d, {"name": pname, **pcoords}
    if best_port:
        new_edges.append(make_edge(
            src["FACILITY_NAME"],
            "mine" if "Mine" in str(src["FACILITY_TYPE"]) else "plant",
            slat, slon,
            best_port["name"], "port", best_port["lat"], best_port["lon"],
            "boron_to_port", "borate", "Boron"))
        boron_edge_count += 1
        print(f"    {src['FACILITY_NAME']:<45} -> {best_port['name']} ({best_dist:.0f} km)")

summary["Boron"] = {"facilities": len(boron_all), "edges": boron_edge_count}
print(f"\n  Boron edges added: {boron_edge_count}")



BORON
  Boron facilities (raw): 20
  Boron facilities (filtered): 3
    Salar de Atacama                                   Mine (active)
    Boric acid and agrochemical plants                 Processing Plant
    Boric acid plant at Antofagasta                    Processing Plant
    Salar de Atacama                              -> Antofagasta (ATI) (215 km)
    Boric acid and agrochemical plants            -> Arica (48 km)
    Boric acid plant at Antofagasta               -> Antofagasta (ATI) (15 km)

  Boron edges added: 3


In [14]:
# SULFURIC ACID: smelter byproduct -> port
# Cu smelters produce H2SO4 as a byproduct. Already partially
# captured in Part 2 (3 downstream edges). Ensure completeness.

section_header("SULFURIC ACID (byproduct)")

# Check existing acid edges
acid_existing = edges[edges["COMMODITIES"].str.contains("Sulfuric|Acid", case=False, na=False)]
print(f"  Existing acid edges: {len(acid_existing)}")

# H2SO4 is consumed domestically by SX-EW plants (acid is a key input).
# Some is exported via Mejillones/Angamos. The existing 3 edges from Part 2
# cover smelter -> port for acid. No additional edges needed unless the
# smelter-to-sxew acid supply chain is also required.
summary["Sulfuric Acid"] = {"existing_edges": len(acid_existing), "new_edges": 0}
print("  Acid is mostly consumed domestically by SX-EW plants.")
print("  Existing smelter->port edges cover export flows.")


SULFURIC ACID (byproduct)
  Existing acid edges: 15
  Acid is mostly consumed domestically by SX-EW plants.
  Existing smelter->port edges cover export flows.


In [15]:
# PROCESSING DEAD-END SWEEP
# Generic 'processing'-stage facilities that received upstream
# mine_to_plant edges but have no downstream edge. These are mostly
# ENAMI custom plants (Centinela oxide, Vallenar, Jose Antonio Moreno,
# Osvaldo Martinez, Manuel Antonio Matta, etc.) that produce
# concentrate or cathode for export.
#
# Strategy: for each dead-end processing plant, determine product form
# from the plant's chain stage or name keywords, then route to the
# nearest matching port.

section_header("PROCESSING DEAD-END SWEEP")

# Collect FROM_NAME set from both existing edges and new_edges
_all_from_names = set(edges["FROM_NAME"].unique())
_all_from_names.update(e["FROM_NAME"] for e in new_edges)

# Collect TO_NAME set to find facilities with upstream
_all_to_names_existing = set(edges["TO_NAME"].unique())
_all_to_names_new = set(e["TO_NAME"] for e in new_edges)
_all_to_names = _all_to_names_existing | _all_to_names_new

# Find processing-stage facilities with upstream but no downstream
proc_plants = inv[
    (inv["CHAIN_STAGE"] == "processing") &
    inv["LATITUD"].notna() &
    inv["FACILITY_NAME"].isin(_all_to_names) &
    ~inv["FACILITY_NAME"].isin(_all_from_names)
].copy()

# Exclude non-mineral processing (cement, methanol, petroleum)
_exclude = "Cement|Methanol|Petroleum|Lime$|Steel"
proc_plants = proc_plants[
    ~proc_plants[comm_col].str.contains(_exclude, case=False, na=False, regex=True)
]

print(f"  Processing dead-end plants to fix: {len(proc_plants)}")

deadend_count = 0
for _, plant in proc_plants.iterrows():
    pname = plant["FACILITY_NAME"]
    plat, plon = plant["LATITUD"], plant["LONGITUD"]
    comms = str(plant.get(comm_col, ""))
    fname_lower = pname.lower()

    # Determine product form and port type from commodity/name
    if "Iron" in comms:
        continue  # handled by iron cell fix
    elif "Iodine" in comms:
        continue  # handled by iodine cell fix
    elif any(kw in fname_lower for kw in ["sx-ew", "leach", "cathode", "catod"]):
        product_form = "cathode_sxew"
        port_type = "cathode"
    elif any(kw in fname_lower for kw in ["smelter", "fundici", "refin"]):
        product_form = "blister"
        port_type = "cathode"
    else:
        # Default: concentrate or general mineral
        product_form = "concentrate"
        port_type = "concentrate"

    # Find nearest port
    assigned_port = None
    # Check dedicated port overrides
    for key, port_name in DEDICATED_PORT.items():
        if key.lower() in fname_lower:
            port = next((p for p in PORTS if p["name"] == port_name), None)
            if port:
                assigned_port = port
                break
    if not assigned_port:
        assigned_port, _ = nearest_port(plat, plon, port_type)

    if assigned_port:
        dist = haversine_km(plat, plon, assigned_port["lat"], assigned_port["lon"])
        # Determine commodity label
        comm_label = "Copper"
        for mineral in ["Gold", "Silver", "Molybdenum", "Zinc", "Lithium", "Nitrate", "Boron"]:
            if mineral in comms:
                comm_label = mineral
                break

        new_edges.append(make_edge(
            pname, "plant", plat, plon,
            assigned_port["name"], "port",
            assigned_port["lat"], assigned_port["lon"],
            "processing_to_port", product_form, comm_label))
        deadend_count += 1
        print(f"    {pname[:50]:<50} -> {assigned_port['name']:<15} [{comm_label}] ({dist:.0f} km)")

summary["Processing dead-ends"] = {"plants": len(proc_plants), "edges": deadend_count}
print(f"\n  Dead-end processing edges added: {deadend_count}")



PROCESSING DEAD-END SWEEP
  Processing dead-end plants to fix: 19
    Centinela oxide and sulfide plant                  -> Angamos         [Copper] (137 km)
    Collahuasi plants                                  -> Patache         [Gold] (167 km)
    Collahuasi plants                                  -> Patache         [Silver] (167 km)
    Facilities near Rancagua                           -> Los Vilos       [Copper] (270 km)
    José Antonio Moreno                                -> Coloso          [Copper] (182 km)
    José Antonio Moreno plant                          -> Coloso          [Gold] (182 km)
    José Antonio Moreno plant                          -> Coloso          [Silver] (182 km)
    Los Bronces Plants                                 -> Los Vilos       [Silver] (179 km)
    Manuel Antonio Matta                               -> Caldera         [Copper] (68 km)
    Manuel Antonio Matta plant                         -> Caldera         [Gold] (68 km)
    Manuel Antonio Ma

In [16]:
# MERGE + DEDUPLICATE + SAVE

section_header("SUMMARY & SAVE")

print(f"New non-copper edges: {len(new_edges)}")
for mineral, info in summary.items():
    edges_n = info.get("edges", info.get("new_edges", 0))
    print(f"  {mineral:<15} {edges_n:>4} edges")

if new_edges:
    new_df = pd.DataFrame(new_edges)
    for col in common_cols:
        if col not in new_df.columns:
            new_df[col] = ""

    edges = pd.concat([edges, new_df[common_cols]], ignore_index=True)

    # Deduplicate
    before_dedup = len(edges)
    edges = edges.drop_duplicates(
        subset=["FROM_NAME", "TO_NAME", "EDGE_TYPE", "COMMODITIES", "PRODUCT_FORM"],
        keep="first")
    n_dupes = before_dedup - len(edges)
    if n_dupes > 0:
        print(f"  Removed {n_dupes} duplicate edges")

print(f"\nTotal edges: {edges_before} -> {len(edges)} (+{len(edges) - edges_before})")
print(f"\nEdge types:")
for et, count in edges["EDGE_TYPE"].value_counts().sort_index().items():
    print(f"  {et:<30} {count:>5}")

print(f"\nCommodities in downstream edges:")
ds = edges[edges["EDGE_TYPE"] != "mine_to_plant"]
for comm, count in ds["COMMODITIES"].value_counts().items():
    print(f"  {comm:<20} {count:>5} downstream edges")

# Save state (capture all variables created during this notebook)
state['inv'] = inv
state['links'] = links
state['edges'] = edges
state['common_cols'] = common_cols
state['smelter_inv_map'] = smelter_inv_map
state['export_df'] = export_df
state['PORT_PRODUCT_MAP'] = PORT_PRODUCT_MAP
state['ports_df'] = ports_df
save_state(state, 5)



SUMMARY & SAVE
New non-copper edges: 218
  Iron              19 edges
  Lithium            3 edges
  Molybdenum        16 edges
  Gold              62 edges
  Silver            33 edges
  Iodine            47 edges
  Nitrate            7 edges
  Zinc               4 edges
  Rhenium            5 edges
  Boron              3 edges
  Sulfuric Acid      0 edges
  Processing dead-ends   19 edges
  Removed 15 duplicate edges

Total edges: 2216 -> 2419 (+203)

Edge types:
  boron_to_port                      3
  concentrate_to_port                6
  concentrate_to_smelter             7
  gold_to_airport                   10
  iodine_to_port                     6
  iron_to_port                       8
  lithium_to_port                    3
  mine_to_mo_plant                  12
  mine_to_plant                   1157
  mine_to_smelter                   78
  mo_to_port                         4
  mo_to_re_plant                     4
  nitrate_to_port                    7
  port_to_country     